# 05. Conformance from `ami_meter_residential` alone (net-only, blind)


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents) if (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from bms_sa_review.ami_analysis.lib import ami_conformance as Conformance
from bms_sa_review.ami_analysis.lib import ami_conformance_plot as ConformancePlot
from bms_sa_review.ami_analysis.lib import ami_pv_phase as PvPhase
from bms_sa_review.ami_analysis.lib import ami_polarity as Polarity
from bms_sa_review.data_query.lib.conformance_metrics import aggregate_sites
from bms_sa_review.shared.ciccada_config import AS4777
from bms_sa_review.synthetic_ami_creation.config import ami_config as Config

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# palette
CATEGORICAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OTHER_COLOR = "#9a9a94"
SEQUENTIAL_HEX = ["#cde2fb", "#86b6ef", "#3987e5", "#2a78d6", "#1c5cab", "#0d366b"]
SEQUENTIAL_CMAP = LinearSegmentedColormap.from_list("seq_blue", SEQUENTIAL_HEX)
ACCENT = CATEGORICAL[0]

In [ ]:
# e.g. ["2024-09", "2024-10"]
# set to None to use every year/month actually present in ami_meter_residential
# SCOPE_MONTHS = ["2024-09", "2024-10", "2024-11"]  
# SCOPE_MONTHS = None
SCOPE_MONTHS = ["2024-09"]
SCOPE_LABEL = "all available months" if SCOPE_MONTHS is None else ", ".join(SCOPE_MONTHS)

# 10% rule (0.10)
SITE_NONCONF_THRESHOLD = AS4777["SITE_CONF_THRESH"]  

# The methodological choices every conformance run below has to be explicit
PRIMARY_RATING_BASIS = "ac_capacity_kw"  
CAPABILITY_PROFILE = "review_corrected"  # AS/NZS 4777.2 Figure 2.1 minimum-capability
                                          # (QCAP) interpretation -- see
                                          # ami_conformance.CAPABILITY_PROFILES'
                                          # docstring for what the two profiles mean

In [ ]:
def scope_sql(months, table_alias=""):
    prefix = f"{table_alias}." if table_alias else ""
    if months is None:
        return "1 = 1"
    pairs = [tuple(int(part) for part in m.split("-")) for m in months]
    clauses = " OR ".join(f"({prefix}year = {y} AND {prefix}month = {m})" for y, m in pairs)
    return f"({clauses})"

### Circuit-polarity correction (applied once, here, to `ami_meter_residential`)

In [ ]:
con = duckdb.connect()

con.sql(f"""
    CREATE OR REPLACE VIEW _ami_meter_residential_raw AS
    SELECT * FROM read_parquet(
        \'{Config.store_path("ami_meter_residential").as_posix()}/dt_month=*/*.parquet\', hive_partitioning=1)
""")

con.sql(f"""
    CREATE OR REPLACE VIEW ami_circuit_polarity AS
    SELECT circuit_id, circuit_polarity
    FROM read_parquet(\'{Config.store_path("ami_circuit_metadata").as_posix()}\')
""")

con.sql(f"""
    CREATE OR REPLACE VIEW ami_meter_residential AS
    {Polarity.build_polarity_corrected_sql("_ami_meter_residential_raw", "ami_circuit_polarity")}
""")

con.sql(f"""
    CREATE OR REPLACE VIEW ami_site_metadata_residential AS
    SELECT * FROM read_parquet(\'{Config.store_path("ami_site_metadata_residential").as_posix()}\')
""")

site_meta = con.sql("SELECT * FROM ami_site_metadata_residential").df()

print(f"{len(site_meta):,} residential sites loaded into site_meta.")
print(f"Scope: {SCOPE_LABEL}. Blind-analysis views ready (ami_meter_residential is "
      "circuit-polarity-corrected -- see markdown above). ami_raw / the full "
      "ami_circuit_metadata (is_pv/circuit_type) are loaded later, in Section 6 "
      "(Evaluation) only.")

polarity_summary = con.sql("""
    SELECT circuit_polarity_applied, count(DISTINCT circuit_id) AS n_circuits, count(*) AS n_rows
    FROM ami_meter_residential
    GROUP BY circuit_polarity_applied
    ORDER BY circuit_polarity_applied
""").df()
print("\ncircuit_polarity correction applied to ami_meter_residential "
      "(P_kw/Q_kvar flipped where circuit_polarity == -1; missing metadata defaults to +1):")
display(polarity_summary)

RATING_BASES = ["ac_capacity_kw", "s_99"]

print("\nMethodology used everywhere below (Sections 4, 5.2, 6):")
display(pd.DataFrame([
    {"setting": "months in scope", "value": SCOPE_LABEL},
    {"setting": "rating bases compared", "value": ", ".join(RATING_BASES)},
    {"setting": "primary rating basis (single-basis results)", "value": PRIMARY_RATING_BASIS},
    {"setting": "Volt-VAr capability profile (QCAP clamp)", "value": CAPABILITY_PROFILE},
    {"setting": "site conformance threshold", "value": SITE_NONCONF_THRESHOLD},
]))


## 1. Phase count vs metadata info on phase

In [ ]:
observed_phase_counts = con.sql("""
    SELECT site_id, count(DISTINCT circuit_id) AS n_circuits_observed
    FROM ami_meter_residential
    GROUP BY site_id
""").df()

phase_check = site_meta[["site_id", "n_load_phases", "n_pv_phases"]].merge(
    observed_phase_counts, on="site_id", how="left",
)

phase_check["n_circuits_observed"] = phase_check["n_circuits_observed"].fillna(0).astype(int)
phase_check["phase_count_mismatch"] = phase_check["n_load_phases"] != phase_check["n_circuits_observed"]

n_mismatch = int(phase_check["phase_count_mismatch"].sum())
print(f"{len(phase_check):,} residential sites checked. "
      f"{n_mismatch:,} ({n_mismatch / len(phase_check):.2%}) have a "
      "load-phase-count mismatch between metadata and what's actually landed.")
phase_check.loc[phase_check.phase_count_mismatch].head(20)


## 2. PV-bearing circuit inference (blind)

- Same approach as previously applied to dnsp analysis initially
- For each site, rank its `ac_load_net` circuits by how much their median`P_kw` differs between a daytime and a nighttime window
- A circuit with PV behind it should show a bigger day/night difference than one without, purely from PV offsetting load at midday. 
- `n_pv_phases` (from metadata install-time information, not inferred from the signal, same as DNSP's `install_phase_count`) sets how many circuits should be selected.


- `DAYTIME_HOURS`/`NIGHTTIME_HOURS` are AEST (fixed +10, no DST, matching this convention). 
- The signature is UNSIGNED (see `ami_pv_phase.py`'s module docstring). `ac_load_net`'s raw `P_kw` sign convention isn't a resolved question project-wide, so this is a weaker
signal than DNSP had, by design, and is exactly why Section 5 checks it against ground truth rather than trusting it on its own.

- **Every one of `site_meta`'s residential sites is mapped, not only the ones
with data in scope.** 
- `circuit_profile` starts from `site_meta` and LEFT-joins the observed signature onto it, so a site with zero `ami_meter_residential` rows in `SCOPE_MONTHS` still gets a row (0 available circuits) instead of silently disappearing. it lands in `insufficient_circuits`, distinguishable below from sites that genuinely have too few circuits despite having data.

- `circuit_id` is cast to pandas' nullable `Int64` right after that merge. 
- Without it, the placeholder `NaN` rows force the whole `circuit_id` column to plain `float64`

In [ ]:
DAYTIME_HOURS = (10, 14)    # AEST, [start, end)
NIGHTTIME_HOURS = (0, 4)    # AEST, [start, end)

circuit_signature = con.sql(f"""
    WITH local_hour AS (
        SELECT site_id, circuit_id, P_kw,
               CAST((EXTRACT(hour FROM t_stamp) + 10) % 24 AS INTEGER) AS hour_aest
        FROM ami_meter_residential
        WHERE {scope_sql(SCOPE_MONTHS)}
    )
    SELECT
        site_id,
        circuit_id,
        count(*) AS n_rows,
        count(P_kw) > 0 AS power_measurement_available,
        median(P_kw) FILTER (
            WHERE hour_aest >= {DAYTIME_HOURS[0]} AND hour_aest < {DAYTIME_HOURS[1]}
        ) AS daytime_median_p_kw,
        median(P_kw) FILTER (
            WHERE hour_aest >= {NIGHTTIME_HOURS[0]} AND hour_aest < {NIGHTTIME_HOURS[1]}
        ) AS nighttime_median_p_kw
    FROM local_hour
    GROUP BY site_id, circuit_id
""").df()

circuit_signature["day_night_delta_kw"] = (
    circuit_signature["daytime_median_p_kw"] - circuit_signature["nighttime_median_p_kw"]
).abs()

sites_with_data_in_scope = set(circuit_signature["site_id"].unique())
n_no_data = len(site_meta) - len(sites_with_data_in_scope)
print(f"{len(sites_with_data_in_scope):,} of {len(site_meta):,} residential sites have "
      f"ami_meter_residential rows in scope ({SCOPE_LABEL}); {n_no_data:,} have none "
      "(a different install/removal window, most likely) and will land in "
      "insufficient_circuits below with 0 available circuits, not because they lack phases.")


circuit_profile = site_meta[["site_id", "n_pv_phases"]].merge(
    circuit_signature, on="site_id", how="left",
)
circuit_profile["circuit_id"] = circuit_profile["circuit_id"].astype("Int64")
circuit_profile["power_measurement_available"] = (
    circuit_profile["power_measurement_available"].fillna(False)
)

pv_mapping_unsigned = PvPhase.derive_circuit_pv_mapping(circuit_profile)

# EITHER direction, while real PV correlates strongly in ONE direction.
hourly_profile = con.sql(f"""
    WITH local_hour AS (
        SELECT site_id, circuit_id, P_kw,
               CAST((EXTRACT(hour FROM t_stamp) + 10) % 24 AS INTEGER) AS local_hour
        FROM ami_meter_residential
        WHERE {scope_sql(SCOPE_MONTHS)}
    )
    SELECT site_id, circuit_id, local_hour, median(P_kw) AS median_p_kw
    FROM local_hour
    GROUP BY site_id, circuit_id, local_hour
""").df()

shape_scores = PvPhase.score_circuits_by_solar_shape(hourly_profile)
circuit_profile = circuit_profile.merge(shape_scores, on=["site_id", "circuit_id"], how="left")

pv_mapping = PvPhase.derive_circuit_pv_mapping(circuit_profile, score_column="solar_shape_score")
pv_mapping["has_meter_data_in_scope"] = pv_mapping["site_id"].isin(sites_with_data_in_scope)

print(f"{len(pv_mapping):,} sites mapped -- every residential site now, not only "
      f"those with data in scope. Confidence breakdown (solar-shape-scored):")
display(pv_mapping.phase_mapping_confidence.value_counts())

print("Of the sites with NO meter data in scope, confidence breakdown "
      "(should be entirely insufficient/not_applicable):")
display(pv_mapping.loc[~pv_mapping.has_meter_data_in_scope, "phase_mapping_confidence"].value_counts())

'''
# Before vs after -- how many sites moved confidence tier, and (more
# importantly this time) how many changed WHICH circuit was selected --
# a tier improving means nothing if it's now confidently pointing at the
# wrong circuit, which is exactly what the pv_sign attempt did silently.
_tier_order = ["high", "medium", "low", "insufficient", "not_applicable"]
_compare = pv_mapping_unsigned[["site_id", "phase_mapping_confidence", "inferred_pv_bearing_circuits"]].merge(
    pv_mapping[["site_id", "phase_mapping_confidence", "inferred_pv_bearing_circuits"]],
    on="site_id", suffixes=("_unsigned", "_shape"),
)
_moved_tier = _compare.loc[
    _compare.phase_mapping_confidence_unsigned != _compare.phase_mapping_confidence_shape
]
_moved_circuit = _compare.loc[
    _compare.inferred_pv_bearing_circuits_unsigned != _compare.inferred_pv_bearing_circuits_shape
]
print(f"{len(_moved_tier):,} sites changed confidence tier; "
      f"{len(_moved_circuit):,} sites changed WHICH circuit(s) were selected.")
display(
    pd.crosstab(
        pd.Categorical(_compare.phase_mapping_confidence_unsigned, categories=_tier_order),
        pd.Categorical(_compare.phase_mapping_confidence_shape, categories=_tier_order),
        rownames=["unsigned (before)"], colnames=["solar-shape (after)"],
    )
)

# Sites where the SELECTED circuit itself changed are worth spot-checking
# individually (with plot_site_day_curves, once you have it defined below)
# -- a tier improving because a different circuit won is only good news if
# that circuit is the one that actually looks like PV.
display(_moved_circuit.merge(
    pv_mapping[["site_id", "n_target_pv_phases", "n_available_circuits"]], on="site_id",
).head(20))
'''

## 3. Cohort gate

Kept: 
- sites with `n_pv_phases > 0` (has PV at all in metadata) AND
- meter data present in scope AND 
- `phase_mapping_confidence` in `{high, medium}` from Section 2. 


In [ ]:
ACCEPTED_CONFIDENCE = ("high", "medium")

cohort = site_meta.merge(pv_mapping, on="site_id", how="left")
cohort["gate_has_pv"] = cohort["n_pv_phases"].fillna(0) > 0
cohort["gate_has_data_in_scope"] = cohort["has_meter_data_in_scope"].fillna(False)
cohort["gate_phase_mapping_confidence"] = cohort["phase_mapping_confidence"].isin(ACCEPTED_CONFIDENCE)
cohort["eligible"] = (
    cohort["gate_has_pv"] & cohort["gate_has_data_in_scope"] & cohort["gate_phase_mapping_confidence"]
)

funnel = pd.DataFrame([
    {"stage": "residential sites", "n_sites": len(cohort)},
    {"stage": "+ has PV (metadata)", "n_sites": int(cohort.gate_has_pv.sum())},
    {"stage": "+ has meter data in scope",
     "n_sites": int((cohort.gate_has_pv & cohort.gate_has_data_in_scope).sum())},
    {"stage": "+ acceptable phase-mapping confidence", "n_sites": int(cohort.eligible.sum())},
])
display(funnel)

eligible = cohort.loc[cohort.eligible].copy()
print(f"{len(eligible):,} sites eligible for the blind conformance pass below.")


In [ ]:
print("Confidence breakdown, by count:")
display(pv_mapping.phase_mapping_confidence.value_counts())

print("\nBroken down by target PV-phase count (n_pv_phases) -- see the note "
      "below on why this matters:")
display(pd.crosstab(pv_mapping.n_target_pv_phases, pv_mapping.phase_mapping_confidence))

# Full per-site table, sorted so each confidence tier groups together.
pv_mapping_view = pv_mapping.sort_values(["phase_mapping_confidence", "site_id"])[
    ["site_id", "n_target_pv_phases", "n_available_circuits", "inferred_pv_bearing_circuits",
     "phase_mapping_method", "phase_mapping_confidence", "phase_mapping_margin_ratio",
     "top_day_night_delta_kw", "has_meter_data_in_scope"]
]
display(pv_mapping_view)

# Just one tier at a time, e.g. to eyeball the "low" sites:
display(pv_mapping_view.loc[pv_mapping_view.phase_mapping_confidence == "low"])

### 3.2. PV Mapping view

In [ ]:
pv_mapping_view[
    (pv_mapping_view["n_available_circuits"] == 3) &
    (pv_mapping_view["n_target_pv_phases"] == 3)
]

In [ ]:
# pv_mapping_view[pv_mapping_view['site_id'] == 13683586]

## 4. Verifying daily plots

In [ ]:
import datetime

def plot_site_day_timeseries(site_id, date):
    """
    date: 'YYYY-MM-DD' (an AEST calendar date) or a date/datetime object.
    One line per circuit_id, P_kw and Q_kvar across that AEST day.
    """
    day = datetime.date.fromisoformat(date) if isinstance(date, str) else date

    # t_stamp is stored such that +10h gives AEST (fixed offset, no DST --
    # same convention as everywhere else in this notebook). So the AEST
    # calendar day [day 00:00, day+1 00:00) is this UTC-ish range:
    start_utc = pd.Timestamp(day) - pd.Timedelta(hours=10)
    end_utc = start_utc + pd.Timedelta(days=1)

    day_data = con.sql(f"""
        SELECT *,
               CAST((EXTRACT(hour FROM t_stamp) + 10) % 24 AS INTEGER)
                   + EXTRACT(minute FROM t_stamp) / 60.0 AS local_hour
        FROM ami_meter_residential
        WHERE site_id = {int(site_id)}
          AND t_stamp >= TIMESTAMP '{start_utc}'
          AND t_stamp <  TIMESTAMP '{end_utc}'
        ORDER BY circuit_id, local_hour
    """).df()

    if day_data.empty:
        print(f"No ami_meter_residential rows for site_id={site_id} on {day} (AEST).")
        return day_data

    row = pv_mapping.loc[pv_mapping.site_id == site_id]
    inferred = set()
    if len(row) and row["inferred_pv_bearing_circuits"].iloc[0]:
        inferred = {int(c) for c in row["inferred_pv_bearing_circuits"].iloc[0].split("|")}

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for i, circuit_id in enumerate(sorted(day_data.circuit_id.unique())):
        g = day_data.loc[day_data.circuit_id == circuit_id]
        is_pv = circuit_id in inferred
        style = dict(
            color=CATEGORICAL[i % len(CATEGORICAL)],
            label=f"circuit {circuit_id}" + (" (inferred PV)" if is_pv else ""),
            linewidth=2.0 if is_pv else 1.3,
            linestyle="-" if is_pv else "--",
        )
        axes[0].plot(g.local_hour, g.P_kw, **style)
        axes[1].plot(g.local_hour, g.Q_kvar, **style)

    axes[0].set_ylabel("P_kw (net)")
    axes[1].set_ylabel("Q_kvar (net)")
    for ax in axes:
        ax.set_xlabel("Local hour (AEST)")
        ax.set_xlim(0, 24)
        ax.axhline(0, color=OTHER_COLOR, linewidth=0.8, zorder=0)
    axes[0].legend(fontsize=8)
    fig.suptitle(f"site_id={site_id} -- {day.isoformat()} (AEST)")
    fig.tight_layout()
    plt.show()

    return day_data

In [ ]:
SITE_ID = 75390466
SITE_ID = 1038687151
SITE_ID = 8761927
SITE_ID = 23391550
# SITE_ID = 777227236
SITE_ID = 53511028

DATE = "2024-09-16"

In [ ]:
plot_site_day_timeseries(SITE_ID, DATE)

In [ ]:
import datetime

def plot_site_day_curves(site_id, date):
    """
    3 stacked subplots sharing the x-axis, height ratios [1, 2, 2] (20/40/40%):
      1. Voltage -- one line per circuit, plus the cross-circuit average when
         there's more than one circuit.
      2. Active power (P_kw) -- one line per circuit.
      3. Reactive power (Q_kvar) -- one line per circuit.
    Returns the underlying day's DataFrame (df_day_meter-equivalent).
    """
    day = datetime.date.fromisoformat(date) if isinstance(date, str) else date
    start_utc = pd.Timestamp(day) - pd.Timedelta(hours=10)  # AEST day -> stored t_stamp range
    end_utc = start_utc + pd.Timedelta(days=1)

    df_day_meter = con.sql(f"""
        SELECT *
        FROM ami_meter_residential
        WHERE site_id = {int(site_id)}
          AND t_stamp >= TIMESTAMP '{start_utc}'
          AND t_stamp <  TIMESTAMP '{end_utc}'
        ORDER BY circuit_id, t_stamp
    """).df()

    if df_day_meter.empty:
        print(f"No ami_meter_residential rows for site_id={site_id} on {day} (AEST).")
        return df_day_meter

    row = pv_mapping.loc[pv_mapping.site_id == site_id]
    inferred = set()
    if len(row) and row["inferred_pv_bearing_circuits"].iloc[0]:
        inferred = {int(c) for c in row["inferred_pv_bearing_circuits"].iloc[0].split("|")}

    circuit_ids = sorted(df_day_meter.circuit_id.unique())
    circuit_colors = {c: CATEGORICAL[i % len(CATEGORICAL)] for i, c in enumerate(circuit_ids)}

    # Create the subplots
    fig, (ax_v, ax_p, ax_q) = plt.subplots(
        3, 1, sharex=True, figsize=(12, 9), constrained_layout=True,
        gridspec_kw={"height_ratios": [1, 2, 2]},
    )

    # --- Voltage: per circuit + cross-circuit average ---------------------- #
    for circuit_id in circuit_ids:
        g = df_day_meter.loc[df_day_meter.circuit_id == circuit_id]
        is_pv = circuit_id in inferred
        ax_v.plot(
            g.t_stamp, g.V,
            label=f"Circuit {circuit_id}" + (" (inferred PV)" if is_pv else ""),
            color=circuit_colors[circuit_id],
            linewidth=2.0 if is_pv else 1.3,
            marker="o", markersize=2,
        )
    if len(circuit_ids) > 1:
        avg_v = df_day_meter.groupby("t_stamp", as_index=False)["V"].mean()
        ax_v.plot(avg_v.t_stamp, avg_v.V, label="Average (all circuits)",
                  color=OTHER_COLOR, linewidth=2.0, linestyle="--")
    ax_v.set_ylabel("Voltage (V)")
    ax_v.set_title(f"site_id={site_id} -- {day.isoformat()} (AEST)")
    ax_v.legend(fontsize=8, loc="upper right")
    ax_v.grid(True, alpha=0.3)

    # --- Active power -------------------------------------------------------- #
    for circuit_id in circuit_ids:
        g = df_day_meter.loc[df_day_meter.circuit_id == circuit_id]
        is_pv = circuit_id in inferred
        ax_p.plot(
            g.t_stamp, g.P_kw,
            label=f"Circuit {circuit_id}" + (" (inferred PV)" if is_pv else ""),
            color=circuit_colors[circuit_id],
            linewidth=2.0 if is_pv else 1.3,
            marker="o", markersize=2,
        )
    ax_p.axhline(0, color=OTHER_COLOR, linewidth=0.8, zorder=0)
    ax_p.set_ylabel("Active power P_kw (net)")
    ax_p.legend(fontsize=8, loc="upper right")
    ax_p.grid(True, alpha=0.3)

    # --- Reactive power -------------------------------------------------------- #
    for circuit_id in circuit_ids:
        g = df_day_meter.loc[df_day_meter.circuit_id == circuit_id]
        is_pv = circuit_id in inferred
        ax_q.plot(
            g.t_stamp, g.Q_kvar,
            label=f"Circuit {circuit_id}" + (" (inferred PV)" if is_pv else ""),
            color=circuit_colors[circuit_id],
            linewidth=2.0 if is_pv else 1.3,
            marker="s", markersize=2,
        )
    ax_q.axhline(0, color=OTHER_COLOR, linewidth=0.8, zorder=0)
    ax_q.set_ylabel("Reactive power Q_kvar (net)")
    ax_q.set_xlabel("Timestamp")
    ax_q.legend(fontsize=8, loc="upper right")
    ax_q.grid(True, alpha=0.3)

    plt.show()
    return df_day_meter

In [ ]:
'''
SITE_ID = 13683586
SITE_ID = 9255455
# SITE_ID = 1604765439
SITE_ID = 276149647

SITE_ID = 276149647
SITE_ID = 13683586

SITE_ID = 2105830595

SITE_ID = 45773670

DATE = "2024-09-15"
'''

In [ ]:
df_day_meter = plot_site_day_curves(
SITE_ID, DATE
)
df_day_meter

In [ ]:
site_meta.loc[site_meta.site_id == SITE_ID].merge(
    pv_mapping.loc[pv_mapping.site_id == SITE_ID], on="site_id", how="left",
)

In [ ]:
plot_columns = [
    "P_kw",
    "Q_kvar",
    "energy_import_kwh",
    "energy_export_kwh",
]

fig, axes = plt.subplots(
    len(plot_columns), 1,
    figsize=(12, 10),
    sharex=True,
    constrained_layout=True,
)

circuit_ids = sorted(df_day_meter["circuit_id"].dropna().unique())
circuit_colors = {
    circuit_id: CATEGORICAL[i % len(CATEGORICAL)]
    for i, circuit_id in enumerate(circuit_ids)
}

for ax, column in zip(axes, plot_columns):
    for circuit_id in circuit_ids:
        circuit_data = (
            df_day_meter.loc[
                df_day_meter["circuit_id"] == circuit_id,
                ["t_stamp", column],
            ]
            .dropna()
            .sort_values("t_stamp")
        )

        ax.plot(
            circuit_data["t_stamp"],
            circuit_data[column],
            label=f"Circuit {circuit_id}",
            color=circuit_colors[circuit_id],
            linewidth=1.5,
        )

    ax.set_ylabel(column)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

axes[-1].set_xlabel("Timestamp")
fig.suptitle(f"Site {SITE_ID} on {DATE}")
plt.show()

In [ ]:
### WILL BREAK IF SITE has 1 PHASE

'''
# Inspect the exact per-circuit day/night signature values that drove
# site 13683586's ranking -- confirms which circuit is the near-tied runner-up.
site_check = circuit_signature.loc[circuit_signature.site_id == SITE_ID].copy()
site_check = site_check.sort_values("day_night_delta_kw", ascending=False)
display(
    site_check[[
        "circuit_id",
        "daytime_median_p_kw",
        "nighttime_median_p_kw",
        "day_night_delta_kw",
    ]]
)

# Recompute the margin_ratio directly to sanity-check against the stored value.
top2 = site_check["day_night_delta_kw"].head(2).tolist()
winner, runner_up = top2[0], top2[1]
print(f"winner={winner:.6f}  runner_up={runner_up:.6f}  "
      f"margin_ratio={(winner - runner_up) / max(abs(winner), 1e-6):.6f}")

# And the fixed (solar-shape) picture for the same site -- should show
# 448410 far ahead of both load circuits, not a near-tie.
display(
    circuit_profile.loc[circuit_profile.site_id == SITE_ID, [
        "circuit_id", "day_night_delta_kw", "solar_shape_score", "solar_shape_sign",
    ]].sort_values("solar_shape_score", ascending=False)
)
'''

## 5. Checking `ami_meter_residential` against ground truth

Given a `SITE_ID` and a `DATE`, plot a 6-panel AS/NZS 4777.2 diagnostic for that one site/day: voltage, reactive power, and active power for every circuit at the site (raw net readings, panels 1/2/4/6), plus (for the inferred PV circuit specifically) its reactive/active power against the Volt-VAr/Volt-Watt bands (panels 3/5), and total apparent power across all circuits.

- Lives in `ami_conformance_plot.py` (`bms_sa_review/ami_analysis/lib`), not pasted here -- ported in shape from the 6-subplot diagnostic in `dnsp_analysis/notebooks/draft.ipynb`, adapted to this project's own net-meter data.
- The Volt-VAr/Volt-Watt bands are NOT re-derived -- `fetch_site_day_conformance` calls `Conformance.build_voltvar_sql`/`build_voltwatt_sql` directly (the same SQL Section 6 below uses), so this plot's bands are guaranteed identical to the fleet-wide methodology, not a parallel implementation that can drift out of sync.
- **Voltage panel colouring** matches `draft.ipynb` exactly: no shaded band below the Volt-VAr ramp-start voltage (V3) -- only the three bands above it (green V3-V1, orange V1-V4, red V4+). An earlier version of this plot incorrectly also shaded a band below V3; fixed.
- **Panel 3's shaded band** no longer blows out in the "not assessable" (|P| < 20% rating) region. `build_voltvar_sql`'s capability clamp relaxes the tolerance band outward to whatever the (near-zero-P) capability floor computes to, which the standard doesn't actually constrain there and which could blow up to an implausibly wide band -- especially with a baseline-corrected P estimate hovering near zero. That region draws a simple, bounded required-Q +/-tolerance band instead, matching how `draft.ipynb`'s own `calculate_voltvar_band` falls back to a plain band whenever a response isn't required.

**Baseline-corrected panels 3/5.** `ami_meter_residential`'s `P_kw`/`Q_kvar` are the `ac_load_net` circuit's own reading -- already net-of-solar as landed, per `ami_dataset_column_reference.xlsx` -- not a pure house-load measurement and not the inverter's own generation/absorption either. Comparing that raw net reading directly against the AS/NZS 4777.2 generator-convention curves can look qualitatively *wrong* even when nothing is broken: reactive-power **absorption** (what the standard requires whenever voltage is high) makes the *net* reading go *more* positive, not less, since absorbing behaves like an extra inductive load on the meter rather than offsetting anything the way generation offsets active-power import (this is exactly what site 276149647 below shows). This was confirmed against real raw data (`corr(Q_load_kvar, Q_pv_kvar) = -0.988` for that site) -- not a polarity issue (both circuits there have `circuit_polarity = 1`).

To recover a signal that's actually comparable to the standard's curves, panels 3/5 now use a **nighttime-baseline estimate** of the inverter's own P/Q (`ami_pv_baseline.py`): since PV output is ~0 overnight, the net reading there is ~= the household's gross (PV-independent) consumption; that per-circuit nighttime median becomes the baseline, and `pv_estimate = nighttime_baseline - net_reading` for both P and Q. This is a real **approximation**, not a measurement -- it assumes the household's own load pattern doesn't shift drastically between night and day. It's controlled by `fetch_site_day_conformance`/`plot_site_day_conformance`'s `use_baseline_correction` (default `True`), `baseline_where_clause` (which rows to aggregate the baseline over -- default the circuit's whole history; scope it to match this notebook's `SCOPE_MONTHS` for consistency, or narrow to the site's own history), and `nighttime_hours` (default AEST `(0, 4)`) parameters. Set `use_baseline_correction=False` to see the old, uncorrected raw-net comparison. The baseline actually subtracted is annotated directly on panels 3/5's titles, and also returned in `pv_day_classified["nighttime_baseline_p_kw"]`/`["nighttime_baseline_q_kvar"]`.

Validate this approximation against Section 6's `ami_raw_phaseseparate`-based "true" PV-only comparison before trusting it past a first-pass check -- it is a way to approximate the inverter's own signal from AMI alone (the whole point of this "blind" notebook), not a substitute for it.

- Panels 1/2/4/6 are unaffected by any of the above -- they always show every circuit's raw net reading, for physical context.
- The x-axis is the raw `t_stamp` (not shifted to AEST) -- add 10h to read local time, same convention as the Section 2/4 daily plots above.
- Pass `pv_circuit_id` explicitly (from `pv_mapping.inferred_pv_bearing_circuits`) -- this module has no opinion of its own about which circuit is PV.

**Below the single-day plot**, a second cell gives the same circuit's Q-vs-V response as a SCATTER across the whole `SCOPE_MONTHS` window (`plot_pv_circuit_vvar_scatter`, ported in shape from `dnsp_analysis/notebooks/voltvar_voltwatt_lib.py`'s `plot_vvar_month_scatter`) -- one point per interval, coloured by `voltvar_status`, against the required-Q curve and its capability-clamped tolerance band at 100% output. This is the better view for "does this circuit track the band overall", since one day can be an outlier; the single-day 6-panel plot above is the better view for "what actually happened, hour by hour, on this day". Same baseline correction applies (`use_baseline_correction=True` by default).

In [ ]:
# PV_CIRCUIT_ID accepts a single circuit id, a "|"-joined multi-circuit
# string (this project's own multi-phase format), or a list/tuple -- no
# need to int() or split it yourself; ConformancePlot normalizes it.
PV_CIRCUIT_ID = pv_mapping.loc[pv_mapping.site_id == SITE_ID, "inferred_pv_bearing_circuits"].iloc[0]

fig, all_circuits_day, pv_day_classified = ConformancePlot.plot_site_day_conformance(
    con, SITE_ID, DATE, PV_CIRCUIT_ID,
    rating_column=PRIMARY_RATING_BASIS,
    capability_profile=CAPABILITY_PROFILE,
    # use_baseline_correction=True by default -- panels 3/5 compare the
    # nighttime-baseline-corrected P/Q estimate, not the raw net reading.
    # Set False to see the old, uncorrected comparison for contrast.
)

print(
    f"Nighttime baseline subtracted -- P: {pv_day_classified['nighttime_baseline_p_kw'].iloc[0]:.3f} kW, "
    f"Q: {pv_day_classified['nighttime_baseline_q_kvar'].iloc[0]:.3f} kvar"
)
print("Volt-VAr status breakdown for this circuit/day:")
display(pv_day_classified.voltvar_status.value_counts())

In [ ]:
VVAR_SCOPE_WHERE = scope_sql(SCOPE_MONTHS)

fig_scatter, pv_scope_classified, _ = ConformancePlot.plot_pv_circuit_vvar_scatter(
    con, SITE_ID, PV_CIRCUIT_ID,
    scope_where=VVAR_SCOPE_WHERE,
    period_label=SCOPE_LABEL,
    rating_column=PRIMARY_RATING_BASIS,
    capability_profile=CAPABILITY_PROFILE,
    # use_baseline_correction=True by default, same as the single-day plot above.
    figsize=(8,6)
)

## 6. Volt-Watt / Volt-VAr conformance, comparing rating bases

- Runs both mechanisms against the net `P_kw`/`Q_kvar`/`V` of each eligible
site's INFERRED PV-bearing circuit(s) (summed when more than one)
- Curve math and classification are `ami_conformance.py` ported from `build_conformance_voltvar.py`/`build_conformance_voltwatt.py`.

Volt-VAr and Volt-Watt are reported SEPARATELY, each two ways:

1. **Fleet-level, the project's own 10% rule** (`AS4777["SITE_CONF_THRESH"]`,
   via `conformance_metrics.aggregate_sites` -- reused, not reimplemented):
   for Volt-VAr, `reduced_nonconf_count = Q_adverse + Q_inactive +
   Q_significant_shortfall` (`Q_near_conformant`/`Q_major_surplus` excluded,
   same as `build_conformance_voltvar.py`); a site is conformant if that sum
   is under 10% of its `capability_assessable` intervals. For Volt-Watt,
   `nonconf_count` is intervals with `nonconformance_voltwatt > 0` over
   `exposed_count` (intervals with `V` above the Volt-Watt threshold).
2. **Interval-level category breakdown** -- the raw share of every interval
   in each of the five Volt-VAr categories (or, for Volt-Watt, exposure share
   and mean nonconformance).

Volt-VAr now applies the Figure 2.1 minimum-capability (QCAP) clamp --
`CAPABILITY_PROFILE` (set in the bootstrap cell, printed in the manifest
above). An interval that fails the clamp's own assessability test (only
possible under `review_corrected`, below 20% of the rating column) shows up
as the sixth status, `Q_not_assessable`, in the category breakdown --
`capability_assessable_count` (the 10% rule's own denominator) already
excludes it, same as `build_conformance_voltvar.py`. Volt-Watt has no
capability-clamp concept in the standard, so it's unaffected by
`CAPABILITY_PROFILE`.


In [ ]:
Q_CATEGORIES = (
    "Q_adverse",                 # wrong direction
    "Q_inactive",                # no response
    "Q_significant_shortfall",   # responded, but far short
    "Q_near_conformant",         # 90-110% of required
    "Q_major_surplus",           # over-response
)
Q_REDUCED_CATEGORIES = ("Q_adverse", "Q_inactive", "Q_significant_shortfall")  # the 10% rule's numerator
Q_NOT_SCORED_STATUSES = ("Q_not_assessable", "Q_not_exposed")  # excluded from the 10% rule's own denominator --
                                                                # Q_not_assessable: QCAP clamp, <20% S_rated (review_corrected)
                                                                # Q_not_exposed: outside the standard's own V<=220/V>=240 bands
                                                                # (the 220-240V deadband, where the curve already requires Q=0)
Q_ALL_STATUSES = Q_CATEGORIES + ("Q_conformant",) + Q_NOT_SCORED_STATUSES

eligible_site_ids = ",".join(str(s) for s in eligible.site_id.tolist())
inferred_map = eligible.set_index("site_id")["inferred_pv_bearing_circuits"]

# One row per (site_id, t_stamp): the eligible site's inferred circuit(s)
# summed -- a multi-circuit PV inference is combined the same way a real net
# meter would present a single logical PV connection point.
pv_bearing_pairs = [
    (site_id, circuit_id)
    for site_id, circuits in inferred_map.items()
    for circuit_id in circuits.split("|") if circuits
]
pv_bearing_df = pd.DataFrame(pv_bearing_pairs, columns=["site_id", "circuit_id"])
pv_bearing_df["circuit_id"] = pv_bearing_df["circuit_id"].astype(int)
con.register("pv_bearing_circuits", pv_bearing_df)

inferred_pv_signal = con.sql(f"""
-- avg(m.V), grouped by (site_id, t_stamp) only -- NOT by V itself.
-- A multi-circuit (multi-phase) PV inference sums P_kw/Q_kvar across
-- circuits, so its voltage must be combined the same way, not left as a
-- GROUP BY key: different phases carry different voltages at the same
-- timestamp, so grouping by V too would fragment one multi-phase PV
-- reading into several single-phase rows (defeating the sum() above) and
-- score each phase's own voltage separately against the Volt-VAr/
-- Volt-Watt thresholds instead of the combined PV output's one voltage.
-- Matches ami_build.build_ami_raw's own V_pv convention (an average
-- across the site's kept PV circuits), so this stays comparable to the
-- true PV-only signal used in Section 7.2.
SELECT m.site_id, m.t_stamp,
        avg(m.V) AS V,
        sum(m.P_kw) AS P_kw,
        sum(m.Q_kvar) AS Q_kvar
FROM ami_meter_residential m
INNER JOIN pv_bearing_circuits c USING (site_id, circuit_id)
WHERE {scope_sql(SCOPE_MONTHS, "m")}
GROUP BY m.site_id, m.t_stamp
""").df()


def site_level_conformance(count_columns, numerator_columns, denominator_column, threshold):
    """One row per site (interval counts already crosstabbed), then apply the
    project's own `aggregate_sites` 10% rule -- shared helper for Volt-VAr
    and Volt-Watt below, and reused again in Section 5 to grade the blind
    pass against the true PV-only signal."""
    site_counts = count_columns.reset_index()
    site_counts["_numerator"] = site_counts[list(numerator_columns)].sum(axis=1)
    return aggregate_sites(
        site_counts, "_numerator", denominator_column, threshold=threshold,
    )


blind_results = {}  # keyed by rating_basis -- reused in Section 5's evaluation
voltvar_aggregate_rows = []
voltvar_category_rows = []
voltwatt_aggregate_rows = []

for rating_basis in RATING_BASES:
    scored = inferred_pv_signal.merge(
        eligible[["site_id", rating_basis]], on="site_id", how="left",
    ).rename(columns={rating_basis: "rating_kw"})
    scored = scored.loc[scored.rating_kw.notna() & (scored.rating_kw > 0)]
    con.register("_scored", scored)

    vvar = con.sql(Conformance.build_voltvar_sql(
        "_scored", rating_column="rating_kw", capability_profile=CAPABILITY_PROFILE,
    )).df()
    vw = con.sql(Conformance.build_voltwatt_sql("_scored", rating_column="rating_kw")).df()
    con.unregister("_scored")

    # --- Volt-VAr -------------------------------------------------------- #
    category_shares = (
        vvar["voltvar_status"].value_counts(normalize=True).reindex(Q_ALL_STATUSES, fill_value=0.0)
    )
    voltvar_category_rows.append({"rating_basis": rating_basis, **category_shares.to_dict()})

    vvar_status_counts = (
        vvar.groupby("site_id")["voltvar_status"].value_counts()
        .unstack(fill_value=0).reindex(columns=Q_ALL_STATUSES, fill_value=0)
    )
    # An interval only counts toward the site-level 10% rule when it is BOTH
    # capability-assessable (Figure 2.1's QCAP gate) AND voltage-exposed (the
    # standard's own V<=220/V>=240 bands -- excludes the 220-240V deadband).
    vvar["voltvar_scored"] = vvar["capability_assessable"] & vvar["voltvar_exposed"]
    vvar_status_counts["scored_count"] = vvar.groupby("site_id")["voltvar_scored"].sum()
    vvar_site_conformance = site_level_conformance(
        vvar_status_counts, Q_REDUCED_CATEGORIES, "scored_count", SITE_NONCONF_THRESHOLD,
    )
    voltvar_aggregate_rows.append({
        "rating_basis": rating_basis,
        "n_sites_assessed": len(vvar_site_conformance),
        "fleet_conformant_pct": 100 * vvar_site_conformance["conformant"].mean(),
        "median_site_nonconf_frac": vvar_site_conformance["nonconf_frac"].median(),
    })

    # --- Volt-Watt --------------------------------------------------------- #
    vw_site_counts = vw.groupby("site_id").agg(
        exposed_count=("nonconformance_voltwatt", "count"),
        nonconforming_count=("nonconformance_voltwatt", lambda s: int((s > 0).sum())),
    )
    vw_site_conformance = site_level_conformance(
        vw_site_counts, ["nonconforming_count"], "exposed_count", SITE_NONCONF_THRESHOLD,
    )
    voltwatt_aggregate_rows.append({
        "rating_basis": rating_basis,
        "n_intervals": len(scored),
        "interval_exposed_share": vw["nonconformance_voltwatt"].notna().mean(),
        "interval_mean_nonconformance_kw": vw["nonconformance_voltwatt"].mean(),
        "n_sites_assessed": len(vw_site_conformance),
        "fleet_conformant_pct": 100 * vw_site_conformance["conformant"].mean(),
        "median_site_nonconf_frac": vw_site_conformance["nonconf_frac"].median(),
    })

    blind_results[rating_basis] = {
        "scored": scored, "vvar": vvar, "vw": vw,
        "vvar_site": vvar_site_conformance, "vw_site": vw_site_conformance,
    }

voltvar_aggregate_by_rating_basis = pd.DataFrame(voltvar_aggregate_rows)
voltvar_category_breakdown_by_rating_basis = pd.DataFrame(voltvar_category_rows)
voltwatt_aggregate_by_rating_basis = pd.DataFrame(voltwatt_aggregate_rows)

print("Volt-VAr -- fleet-level 10% rule (site conformant if reduced-nonconformance "
      "< 10% of capability-assessable intervals):")
display(voltvar_aggregate_by_rating_basis)

print("Volt-VAr -- interval-level category breakdown:")
display(voltvar_category_breakdown_by_rating_basis)

print("Volt-Watt -- fleet-level 10% rule and interval-level exposure:")
display(voltwatt_aggregate_by_rating_basis)


## 7. Evaluation -- ground truth, separated

Everything above never touched `ami_circuit_metadata.is_pv`/`circuit_type`,
`ami_raw`, or `ami_raw_phaseseparate`. This section does, ONLY to grade the
blind pipeline above -- never to feed it. Two different questions, kept
separate:

**7.1 -- Did we pick the right circuit(s)?** A phase-inference sanity check:
does the inferred circuit's own net `P_kw` move with the true PV curve? This
does NOT grade conformance -- a site can have the right circuit and still get
a different conformance verdict than the truth, because the circuit signal
is net-of-load, not PV-only.

**7.2 -- How much does using the net (blind) signal change the conformance
answer, versus running the exact same `ami_conformance.py` classification on
the true, already-separated PV-only signal (`ami_raw.P_pv_kw`/
`Q_pv_kvar`/`V_pv`)?** This is the "aggregated vs. non-aggregated" comparison
-- same curve math, same thresholds, same 10% rule, two different inputs:
the blind net-meter estimate from Section 4, and the true PV-only signal
this project's pipeline already knows but a real DNSP never would.


In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW ami_raw AS
    SELECT * FROM read_parquet(
        \'{Config.store_path("ami_raw").as_posix()}/dt_month=*/*.parquet\', hive_partitioning=1)
""")
print("ami_raw view ready -- ground truth, used only from here on.")


### 7.1 Phase-inference sanity check

Two independent checks, kept separate because they answer different questions:

**(a) Phase-count check** -- `inferred_pv_bearing_circuits` is a set of
`ac_load_net` circuit_ids (the blind algorithm only ever sees load circuits
-- `ami_meter_residential` never carries a `pv_site_net` row at all, see
Section 0). A `pv_site_net` circuit_id can therefore NEVER equal one of
these -- they're disjoint circuit_type populations by construction, and
`ami_metadata.py`'s own docs say the one field that might have linked a
specific load circuit to the PV circuit on its same phase (`m_id`) is
explicitly "NOT reliable for phase-grouping." So there is no direct
circuit-to-circuit ground truth to check here -- the one thing that IS
directly answerable is COUNT: did we infer the right NUMBER of PV-bearing
phases (e.g. found all 3 phases on a 3-phase system, or only 1 of them)?
Broken down by the site's TRUE phase configuration (ground truth from
`ami_raw_phaseseparate`, same convention notebook 99 uses).

**(b) Signal correlation** -- for every site (whichever circuit(s) got
picked), does that circuit's own net `P_kw` move the way true PV generation
does? This is the check for whether the right PHASE was picked, which (a)
can't answer directly. `ami_raw.P_pv_kw` is the site-level TRUE PV
generation, properly signed, positive while generating; the inferred
circuit's net `P_kw` is expected to move in the OPPOSITE direction
(net-of-PV: more true generation -> lower net reading), so a correlation
near -1 is the "looks right" signature. Two refinements versus the first
pass at this:

- Restricted to solar hours only (`PvPhase.DEFAULT_SUNRISE_HOUR` to
  `DEFAULT_SUNSET_HOUR`, AEST -- the same daytime window the blind
  phase-mapping algorithm's own `solar_shape_score` uses), so a whole-day
  average dominated by nighttime load-only hours doesn't dilute the daytime
  relationship.
- Reported both "as is" (raw net `P_kw`) and nighttime-baseline-corrected
  (subtracting each site's own nighttime-median `P_kw` -- ONE stable value
  per site over the whole scope, not recomputed per day, matching this
  project's own established convention in `ami_pv_baseline
  .build_nighttime_baseline_sql`'s docstring).

Circuit-polarity correction is NOT redone here -- it was already applied,
once, fleet-wide, to `ami_meter_residential` itself (the bootstrap cell
above), and `inferred_pv_signal` is built from that already-corrected view.
A positive correlation here can still mean several different things (wrong
circuit entirely, right circuit but a bad polarity value for it, or a real
but weak/swamped relationship) -- the `count_matched` column joined into
the outlier list below at least tells you whether the phase COUNT was
right for that site, as one piece of context (it can't confirm the specific
phase was right, per (a) above).


In [ ]:
# TRUE circuit identity, from ami_raw_phaseseparate -- the one Phase-5 table
# that keeps BOTH ac_load_net and pv_site_net rows at circuit grain, tagged
# by circuit_type (same source/convention as notebook 99's Section 1).
# Loaded here, not earlier -- per this section's own rule, nothing above
# this point has touched the true labels.
con.sql(f"""
    CREATE OR REPLACE VIEW ami_raw_phaseseparate AS
    SELECT * FROM read_parquet(
        '{Config.store_path("ami_raw_phaseseparate").as_posix()}/dt_month=*/*.parquet', hive_partitioning=1)
""")

true_circuits = con.sql(f"""
    SELECT DISTINCT site_id, circuit_id, circuit_type
    FROM ami_raw_phaseseparate
    WHERE {scope_sql(SCOPE_MONTHS)} AND site_id IN ({eligible_site_ids})
""").df()

true_pv_ids = (
    true_circuits.loc[true_circuits.circuit_type == "pv_site_net"]
    .groupby("site_id")["circuit_id"].apply(lambda s: frozenset(int(c) for c in s))
)
true_load_ids = (
    true_circuits.loc[true_circuits.circuit_type == "ac_load_net"]
    .groupby("site_id")["circuit_id"].apply(lambda s: frozenset(int(c) for c in s))
)
inferred_ids = pv_bearing_df.groupby("site_id")["circuit_id"].apply(lambda s: frozenset(int(c) for c in s))

# Same phase-combo bucketing as notebook 99's PHASE_COMBOS, from the TRUE
# circuit counts -- NOT site_meta.n_pv_phases/n_load_phases (that
# metadata's own scope-mismatch caveat, per ami_metadata.py).
PHASE_COMBOS = {
    "3-phase PV + 3-phase Load": (3, 3),
    "1-phase PV + 3-phase Load": (1, 3),
    "3-phase PV + 1-phase Load": (3, 1),
    "1-phase PV + 1-phase Load": (1, 1),
}
PHASE_COMBO_ORDER = list(PHASE_COMBOS) + ["other"]


def _phase_combo_label(n_pv: int, n_load: int) -> str:
    for label, (pv, load) in PHASE_COMBOS.items():
        if n_pv == pv and n_load == load:
            return label
    return "other"


# `count_matched` is the only DIRECT, non-proxy ground-truth check possible
# here -- see the markdown above for why a literal circuit-id match is
# structurally impossible (disjoint circuit_type populations, no reliable
# phase-linking key). It checks COUNT only, not which specific circuit(s).
identity_rows = []
for site_id in eligible.site_id:
    true_ids = true_pv_ids.get(site_id, frozenset())
    load_ids = true_load_ids.get(site_id, frozenset())
    guess_ids = inferred_ids.get(site_id, frozenset())
    identity_rows.append({
        "site_id": site_id,
        "n_true_pv_circuits": len(true_ids),
        "n_true_load_circuits": len(load_ids),
        "n_inferred_circuits": len(guess_ids),
        "count_matched": len(guess_ids) == len(true_ids),
        "phase_combo": _phase_combo_label(len(true_ids), len(load_ids)),
    })
identity_check = pd.DataFrame(identity_rows).merge(
    eligible[["site_id", "phase_mapping_confidence"]], on="site_id", how="left"
)

print(f"Cohort-level PHASE-COUNT match: {identity_check.count_matched.mean():.1%} of "
      f"{len(identity_check):,} eligible sites guessed the right NUMBER of PV-bearing "
      f"circuits ({int(identity_check.count_matched.sum()):,} sites). This checks COUNT "
      "only, not which specific circuit(s) -- see the markdown above for why; Section "
      "7.1(b)'s correlation is the check for whether the right phase(s) were actually picked.")
print("\nBy TRUE phase combination:")
display(
    identity_check.groupby("phase_combo")
    .agg(n_sites=("site_id", "count"), n_count_matched=("count_matched", "sum"),
         count_match_rate=("count_matched", "mean"))
    .reindex(PHASE_COMBO_ORDER)
)
print("\nBy blind confidence x TRUE phase combination (phase-count match rate):")
display(
    identity_check.groupby(["phase_combo", "phase_mapping_confidence"])["count_matched"]
    .mean().unstack(fill_value=np.nan).reindex(PHASE_COMBO_ORDER)
)


In [ ]:
# HYPOTHESIS CHECK -- does `device_id` cross-link a load circuit to the PV
# circuit on the SAME phase? `ami_phase3_status_and_phase4_plan.md` already
# confirmed `device_id` (not `m_id`, not `voltage_class`) is the validated
# key for grouping multiple circuit_ids of the SAME circuit_type into one
# physical device's phases (e.g. three `pv_site_net` circuit_ids under one
# `device_id` are three genuine PV phases, not duplicates). What it did NOT
# establish is whether a load circuit's `device_id` and its co-phase PV
# circuit's `device_id` are the SAME value -- that's what would make this a
# real cross-type phase link, and it's cheap to check directly.
device_ids_by_type = con.sql(f"""
    SELECT DISTINCT site_id, circuit_id, circuit_type, device_id
    FROM ami_raw_phaseseparate
    WHERE {scope_sql(SCOPE_MONTHS)} AND site_id IN ({eligible_site_ids})
""").df()

pv_device_ids = (
    device_ids_by_type.loc[device_ids_by_type.circuit_type == "pv_site_net"]
    .groupby("site_id")["device_id"].apply(lambda s: frozenset(s.dropna()))
)
load_device_ids = (
    device_ids_by_type.loc[device_ids_by_type.circuit_type == "ac_load_net"]
    .groupby("site_id")["device_id"].apply(lambda s: frozenset(s.dropna()))
)

overlap_rows = []
for site_id in eligible.site_id:
    pv_devs = pv_device_ids.get(site_id, frozenset())
    load_devs = load_device_ids.get(site_id, frozenset())
    overlap_rows.append({
        "site_id": site_id,
        "n_pv_devices": len(pv_devs),
        "n_load_devices": len(load_devs),
        "n_shared_device_ids": len(pv_devs & load_devs),
        "any_overlap": bool(pv_devs & load_devs),
    })
device_overlap = pd.DataFrame(overlap_rows)

both_present = (device_overlap.n_pv_devices > 0) & (device_overlap.n_load_devices > 0)
n_with_both = int(both_present.sum())
n_overlap = int(device_overlap.loc[both_present, "any_overlap"].sum())

print(f"{n_with_both:,} eligible sites have BOTH a PV and a load device_id present in "
      "ami_raw_phaseseparate.")
if n_with_both:
    print(f"{n_overlap:,} of those ({n_overlap / n_with_both:.1%}) share at least one "
          "device_id between their PV and load circuits.")
    print("\n>= 90% overlap would mean device_id IS a real cross-type phase link -- worth "
          "rebuilding the identity check on top of it instead of the count-only version. "
          "Near 0% means it's NOT (PV and load circuits are numbered in separate device_id "
          "spaces), and the voltage-correlation approach is the next thing to try.")
else:
    print("No eligible site has both present -- can't test the hypothesis on this cohort.")
display(device_overlap.loc[both_present].head(10))


In [ ]:
# STRICTER TEST -- does device_id resolve INDIVIDUAL phase pairs, or does it
# only prove co-location (one shared device_id covering an entire 3-phase
# installation)? The 95.9% "any overlap" figure above can't distinguish these:
# a single-phase site (1 PV device, 1 load device) satisfies "any_overlap"
# almost by construction, and per the project's own docs most 3-phase LOAD
# sites already have all 3 load circuit_ids under ONE device_id -- so even a
# 3-phase site could show n_load_devices=1, in which case device_id carries no
# per-phase information at all, regardless of whether it "overlaps".
#
# Two things to check directly:
#   1. Do device_id counts actually exceed 1 at 3-phase sites in this fleet?
#      If not, device_id is device-grained, not phase-grained, and can't be
#      used to resolve individual phases no matter how well it overlaps.
#   2. Among sites where it's even possible to test (both PV and load are
#      multi-device), does the full set of device_ids match exactly
#      (n_shared_device_ids == n_pv_devices == n_load_devices)?

device_overlap_by_combo = device_overlap.merge(
    identity_check[["site_id", "phase_combo"]], on="site_id", how="left"
)

n_pv_multi = int((device_overlap_by_combo.n_pv_devices > 1).sum())
n_load_multi = int((device_overlap_by_combo.n_load_devices > 1).sum())
print(f"Sites with n_pv_devices > 1:   {n_pv_multi:,} / {len(device_overlap_by_combo):,}")
print(f"Sites with n_load_devices > 1: {n_load_multi:,} / {len(device_overlap_by_combo):,}")
print(
    "\nIf both of these are near zero even within the 3H+3PV combo, device_id "
    "is device-grained (one id per installation) rather than phase-grained, "
    "and can't resolve individual phase pairs -- confirming the project docs' "
    "prior finding for load circuits and extending it to PV circuits."
)

summary_by_combo = (
    device_overlap_by_combo.groupby("phase_combo")
    .agg(
        n_sites=("site_id", "size"),
        mean_n_pv_devices=("n_pv_devices", "mean"),
        mean_n_load_devices=("n_load_devices", "mean"),
        any_overlap_rate=("any_overlap", "mean"),
    )
    .reindex(PHASE_COMBO_ORDER)
)

both_multi = (device_overlap_by_combo.n_pv_devices > 1) & (device_overlap_by_combo.n_load_devices > 1)
device_overlap_by_combo["full_match"] = (
    (device_overlap_by_combo.n_shared_device_ids == device_overlap_by_combo.n_pv_devices)
    & (device_overlap_by_combo.n_shared_device_ids == device_overlap_by_combo.n_load_devices)
    & (device_overlap_by_combo.n_pv_devices > 0)
)
full_match_rate_multi = (
    device_overlap_by_combo.loc[both_multi]
    .groupby("phase_combo")["full_match"].mean()
    .reindex(PHASE_COMBO_ORDER)
)
summary_by_combo["n_sites_both_multi_device"] = (
    device_overlap_by_combo.loc[both_multi].groupby("phase_combo").size().reindex(PHASE_COMBO_ORDER)
)
summary_by_combo["full_match_rate_when_both_multi"] = full_match_rate_multi

print(f"\nSites where BOTH pv and load are multi-device (the only sites this stricter "
      f"test can actually run on): {int(both_multi.sum()):,}")
display(summary_by_combo)

In [ ]:
SOLAR_HOURS = (int(PvPhase.DEFAULT_SUNRISE_HOUR), int(PvPhase.DEFAULT_SUNSET_HOUR))  # (6, 18) AEST
NIGHTTIME_HOURS = (0, 4)  # AEST -- same window as ami_pv_baseline.build_nighttime_baseline_sql's own default

# TRUE PV generation, restricted to solar hours only -- nighttime is
# guaranteed P_pv_kw == 0, so including it would just dilute the daytime
# relationship with a flat, uninformative stretch on both sides.
true_pv_daily = con.sql(f"""
    SELECT site_id, date_trunc('day', t_stamp) AS day, avg(P_pv_kw) AS true_pv_kw
    FROM ami_raw
    WHERE {scope_sql(SCOPE_MONTHS)} AND site_id IN ({eligible_site_ids})
      AND CAST((EXTRACT(hour FROM t_stamp) + 10) % 24 AS INTEGER) >= {SOLAR_HOURS[0]}
      AND CAST((EXTRACT(hour FROM t_stamp) + 10) % 24 AS INTEGER) <  {SOLAR_HOURS[1]}
    GROUP BY site_id, date_trunc('day', t_stamp)
""").df()

# Inferred circuit's own net P_kw -- computed ENTIRELY inside DuckDB this
# time, not pulled into pandas as a 150M+-row interval-level frame first.
# The previous version pulled `inferred_pv_signal` via `.df()` (already
# several GB: site_id/t_stamp/V/P_kw/Q_kvar x ~156M rows) and then tried to
# derive one more column on top of it -- on this machine there was no
# memory headroom left for even a single ~600MB temporary array, regardless
# of how it was computed (`pd.to_datetime` or plain `.dt.hour`, same
# failure either way). DuckDB is a columnar, spill-to-disk engine built
# for exactly this kind of scan+aggregate; doing the hour math and the
# GROUP BY there, then only pulling back the much smaller (site, day) or
# (site,)-grain result, avoids ever materialising the giant intermediate
# in Python at all. This re-derives the same combined-PV-circuit signal as
# `inferred_pv_signal` (cell above, reused by Section 6) via the identical
# JOIN -- it does not depend on or modify that dataframe.
_inferred_signal_sql = f"""
    SELECT m.site_id, m.t_stamp,
           sum(m.P_kw) AS P_kw,
           CAST((EXTRACT(hour FROM m.t_stamp) + 10) % 24 AS INTEGER) AS hour_aest
    FROM ami_meter_residential m
    INNER JOIN pv_bearing_circuits c USING (site_id, circuit_id)
    WHERE {scope_sql(SCOPE_MONTHS, "m")}
    GROUP BY m.site_id, m.t_stamp
"""

# ONE nighttime-median baseline per site, over the WHOLE scope -- not
# recomputed per day (see the markdown above).
nighttime_baseline = con.sql(f"""
    SELECT site_id, median(P_kw) AS nighttime_baseline_p_kw
    FROM ({_inferred_signal_sql}) s
    WHERE hour_aest >= {NIGHTTIME_HOURS[0]} AND hour_aest < {NIGHTTIME_HOURS[1]}
    GROUP BY site_id
""").df()

# Daily mean over solar hours only, on the raw net P_kw -- the baseline
# SUBTRACTION still happens after this daily aggregation (on this small
# pandas table), not before it: since the baseline is one constant per
# site, mean(P_kw - baseline) == mean(P_kw) - baseline exactly, so nothing
# about the result changes.
inferred_pv_daily = con.sql(f"""
    SELECT site_id, date_trunc('day', t_stamp) AS day, avg(P_kw) AS P_kw_raw
    FROM ({_inferred_signal_sql}) s
    WHERE hour_aest >= {SOLAR_HOURS[0]} AND hour_aest < {SOLAR_HOURS[1]}
    GROUP BY site_id, date_trunc('day', t_stamp)
""").df()

inferred_pv_daily = inferred_pv_daily.merge(nighttime_baseline, on="site_id", how="left")
inferred_pv_daily["nighttime_baseline_p_kw"] = inferred_pv_daily["nighttime_baseline_p_kw"].fillna(0.0)
inferred_pv_daily["P_kw_baseline_corrected"] = (
    inferred_pv_daily["P_kw_raw"] - inferred_pv_daily["nighttime_baseline_p_kw"]
)

phase_check_evaluation = inferred_pv_daily.merge(true_pv_daily, on=["site_id", "day"], how="inner")

# Tidy/long format -- one row per (site_id, day, variant) -- so the
# per-variant correlation, summary table, and both plots below are all the
# same shape of computation instead of duplicating "raw" vs
# "baseline_corrected" by hand everywhere.
phase_check_long = phase_check_evaluation.melt(
    id_vars=["site_id", "day", "true_pv_kw"],
    value_vars=["P_kw_raw", "P_kw_baseline_corrected"],
    var_name="variant", value_name="inferred_pv_kw",
)
phase_check_long["variant"] = phase_check_long["variant"].map({
    "P_kw_raw": "raw", "P_kw_baseline_corrected": "baseline_corrected",
})
phase_check_long = phase_check_long.merge(
    identity_check[["site_id", "phase_combo"]], on="site_id", how="left"
)

correlation_by_site = (
    phase_check_long.groupby(["site_id", "variant"])
    .apply(lambda g: g["inferred_pv_kw"].corr(g["true_pv_kw"]) if len(g) > 1 else np.nan)
    .rename("net_vs_true_pv_correlation")
    .reset_index()
    .merge(eligible[["site_id", "phase_mapping_confidence"]], on="site_id", how="left")
    .merge(identity_check[["site_id", "phase_combo", "count_matched"]], on="site_id", how="left")
)

print(f"{correlation_by_site.net_vs_true_pv_correlation.notna().sum():,} of "
      f"{len(correlation_by_site):,} (site, variant) pairs have a computable correlation "
      f"(solar hours {SOLAR_HOURS[0]}-{SOLAR_HOURS[1]}h AEST only).")
print("\nBy variant x confidence:")
display(
    correlation_by_site.groupby(["variant", "phase_mapping_confidence"])["net_vs_true_pv_correlation"]
    .describe()
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, variant in zip(axes, ["raw", "baseline_corrected"]):
    sub = phase_check_long.loc[phase_check_long.variant == variant]
    sns.regplot(
        data=sub, x="true_pv_kw", y="inferred_pv_kw", ax=ax,
        scatter_kws={"s": 6, "alpha": 0.12, "color": ACCENT},
        line_kws={"color": "black", "linewidth": 1.5},
    )
    r = sub["true_pv_kw"].corr(sub["inferred_pv_kw"])
    ax.set_title(f"{variant} (pooled r = {r:.2f}, n = {len(sub):,} site-days)")
    ax.set_xlabel("True site P_pv_kw (daily mean, solar hours)")
    ax.set_ylabel("Inferred circuit net P_kw (daily mean, solar hours)")
sns.despine()
fig.suptitle(f"Cohort: inferred vs. true PV, solar hours only ({SOLAR_HOURS[0]}-{SOLAR_HOURS[1]}h AEST)")
fig.tight_layout()


In [ ]:
present_combos = [c for c in PHASE_COMBO_ORDER if c in phase_check_long.phase_combo.unique()]

fig, axes = plt.subplots(
    2, len(present_combos), figsize=(3.4 * len(present_combos), 6.6), sharex=False, sharey=False,
)
if len(present_combos) == 1:
    axes = axes.reshape(2, 1)

for row, variant in enumerate(["raw", "baseline_corrected"]):
    for col, combo in enumerate(present_combos):
        ax = axes[row, col]
        sub = phase_check_long.loc[
            (phase_check_long.variant == variant) & (phase_check_long.phase_combo == combo)
        ]
        if len(sub) < 2:
            ax.set_title(f"{combo}\n(n={len(sub)}, too few)", fontsize=9)
            ax.axis("off")
            continue
        sns.regplot(
            data=sub, x="true_pv_kw", y="inferred_pv_kw", ax=ax,
            scatter_kws={"s": 5, "alpha": 0.15, "color": ACCENT},
            line_kws={"color": "black", "linewidth": 1.2},
        )
        r = sub["true_pv_kw"].corr(sub["inferred_pv_kw"])
        n_sites = sub.site_id.nunique()
        ax.set_title(f"{combo}\n{variant}, r={r:.2f}, {n_sites:,} sites", fontsize=9)
        ax.set_xlabel("True P_pv_kw" if row == 1 else "")
        ax.set_ylabel("Inferred net P_kw" if col == 0 else "")
sns.despine()
fig.suptitle(f"By TRUE phase combination -- solar hours only ({SOLAR_HOURS[0]}-{SOLAR_HOURS[1]}h AEST)")
fig.tight_layout()


In [ ]:
# IQR outliers + wrong-sign flag, computed on the baseline-corrected,
# solar-hours-only correlation (the "best" version) -- same Tukey's-rule
# method as before, now cross-referenced against the DIRECT circuit-identity
# match from above.
outlier_source = correlation_by_site.loc[correlation_by_site.variant == "baseline_corrected"].copy()
outlier_order = [c for c in ("high", "medium", "low", "insufficient")
                 if c in outlier_source.phase_mapping_confidence.unique()]


def _iqr_bounds(s: pd.Series) -> tuple[float, float]:
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


bounds = {
    conf: _iqr_bounds(outlier_source.loc[outlier_source.phase_mapping_confidence == conf,
                                          "net_vs_true_pv_correlation"])
    for conf in outlier_order
}
outlier_source["_lo"], outlier_source["_hi"] = zip(*outlier_source.phase_mapping_confidence.map(bounds))
is_iqr_outlier = (
    (outlier_source.net_vs_true_pv_correlation < outlier_source["_lo"])
    | (outlier_source.net_vs_true_pv_correlation > outlier_source["_hi"])
)
is_wrong_sign = (
    outlier_source.phase_mapping_confidence.isin(["high", "medium"])
    & (outlier_source.net_vs_true_pv_correlation > 0)
)
outlier_source["outlier_type"] = np.select(
    [is_iqr_outlier & is_wrong_sign, is_iqr_outlier, is_wrong_sign],
    ["iqr_and_wrong_sign", "iqr_outlier", "wrong_sign"],
    default="",
)
outliers = (
    outlier_source.loc[is_iqr_outlier | is_wrong_sign,
                        ["site_id", "phase_mapping_confidence", "phase_combo", "count_matched",
                         "net_vs_true_pv_correlation", "outlier_type"]]
    .sort_values(["phase_mapping_confidence", "net_vs_true_pv_correlation"])
    .reset_index(drop=True)
)

print(f"{len(outliers):,} outlier sites flagged on the baseline-corrected, solar-hours correlation "
      f"({int(is_iqr_outlier.sum()):,} IQR outliers, {int(is_wrong_sign.sum()):,} wrong-sign, "
      f"{int((is_iqr_outlier & is_wrong_sign).sum()):,} both).")
print(f"Of those, {int(outliers.count_matched.sum()):,} actually had the right PHASE COUNT "
      "-- worth a closer look, since a right-count-but-bad-correlation site points at something "
      "other than a missed phase (e.g. a genuinely wrong circuit_polarity value for that circuit, "
      "or the right count but the wrong specific circuit(s)).")
print("\nBreakdown by confidence x outlier_type:")
display(outliers.groupby(["phase_mapping_confidence", "outlier_type"]).size().unstack(fill_value=0))

OUTLIERS_CSV = "phase_mapping_confidence_outliers.csv"
outliers.to_csv(OUTLIERS_CSV, index=False)
print(f"\nWritten: {OUTLIERS_CSV}")
outliers.head(20)


### 7.2 Blind (net-meter) vs. true (PV-only) conformance

Runs the exact same `Conformance.build_voltvar_sql`/`build_voltwatt_sql`, the
same `CAPABILITY_PROFILE`, and the exact same site-level 10% rule from
Section 4, but on `ami_raw`'s already-separated `P_pv_kw`/`Q_pv_kvar`/`V_pv`
instead of the inferred net circuit signal -- everything else (rating bases,
capability profile, thresholds, eligible-site scope) held identical, so any
difference in the answer is attributable to net-vs-true alone.


In [ ]:
true_pv_signal = con.sql(f"""
    SELECT site_id, t_stamp, V_pv AS V, P_pv_kw AS P_kw, Q_pv_kvar AS Q_kvar
    FROM ami_raw
    WHERE {scope_sql(SCOPE_MONTHS)} AND site_id IN ({eligible_site_ids})
""").df()

true_results = {}
for rating_basis in RATING_BASES:
    scored_true = true_pv_signal.merge(
        eligible[["site_id", rating_basis]], on="site_id", how="left",
    ).rename(columns={rating_basis: "rating_kw"})
    scored_true = scored_true.loc[scored_true.rating_kw.notna() & (scored_true.rating_kw > 0)]
    con.register("_scored_true", scored_true)

    true_vvar = con.sql(Conformance.build_voltvar_sql(
        "_scored_true", rating_column="rating_kw", capability_profile=CAPABILITY_PROFILE,
    )).df()
    true_vw = con.sql(Conformance.build_voltwatt_sql("_scored_true", rating_column="rating_kw")).df()
    con.unregister("_scored_true")

    true_vvar_status_counts = (
        true_vvar.groupby("site_id")["voltvar_status"].value_counts()
        .unstack(fill_value=0).reindex(columns=Q_ALL_STATUSES, fill_value=0)
    )
    true_vvar["voltvar_scored"] = true_vvar["capability_assessable"] & true_vvar["voltvar_exposed"]
    true_vvar_status_counts["scored_count"] = (
        true_vvar.groupby("site_id")["voltvar_scored"].sum()
    )
    true_vvar_site = site_level_conformance(
        true_vvar_status_counts, Q_REDUCED_CATEGORIES, "scored_count", SITE_NONCONF_THRESHOLD,
    )

    true_vw_site_counts = true_vw.groupby("site_id").agg(
        exposed_count=("nonconformance_voltwatt", "count"),
        nonconforming_count=("nonconformance_voltwatt", lambda s: int((s > 0).sum())),
    )
    true_vw_site = site_level_conformance(
        true_vw_site_counts, ["nonconforming_count"], "exposed_count", SITE_NONCONF_THRESHOLD,
    )

    true_results[rating_basis] = {
        "vvar": true_vvar, "vw": true_vw, "vvar_site": true_vvar_site, "vw_site": true_vw_site,
    }

print(f"True per-site conformance computed for {len(RATING_BASES)} rating bases, "
      f"{SCOPE_LABEL}, {len(eligible):,} eligible sites.")


In [ ]:
agreement_rows = []
for rating_basis in RATING_BASES:
    blind = blind_results[rating_basis]
    true = true_results[rating_basis]

    # Volt-VAr -- interval-level category agreement (inner join on shared timestamps).
    vvar_intervals = blind["vvar"].merge(
        true["vvar"][["site_id", "t_stamp", "voltvar_status"]].rename(
            columns={"voltvar_status": "true_voltvar_status"}),
        on=["site_id", "t_stamp"], how="inner",
    )
    voltvar_interval_agreement = (vvar_intervals.voltvar_status == vvar_intervals.true_voltvar_status).mean()

    # Volt-VAr -- site-level 10% rule agreement.
    vvar_sites = (
        blind["vvar_site"][["site_id", "conformant"]].rename(columns={"conformant": "blind_conformant"})
        .merge(true["vvar_site"][["site_id", "conformant"]].rename(columns={"conformant": "true_conformant"}),
               on="site_id", how="inner")
    )
    voltvar_site_agreement = (vvar_sites.blind_conformant == vvar_sites.true_conformant).mean()

    # Volt-Watt -- interval-level "is this interval nonconforming" agreement.
    vw_intervals = blind["vw"].merge(
        true["vw"][["site_id", "t_stamp", "nonconformance_voltwatt"]].rename(
            columns={"nonconformance_voltwatt": "true_nonconformance_voltwatt"}),
        on=["site_id", "t_stamp"], how="inner",
    )
    vw_intervals["blind_nonconforming"] = vw_intervals["nonconformance_voltwatt"].fillna(0) > 0
    vw_intervals["true_nonconforming"] = vw_intervals["true_nonconformance_voltwatt"].fillna(0) > 0
    voltwatt_interval_agreement = (vw_intervals.blind_nonconforming == vw_intervals.true_nonconforming).mean()

    # Volt-Watt -- site-level 10% rule agreement.
    vw_sites = (
        blind["vw_site"][["site_id", "conformant"]].rename(columns={"conformant": "blind_conformant"})
        .merge(true["vw_site"][["site_id", "conformant"]].rename(columns={"conformant": "true_conformant"}),
               on="site_id", how="inner")
    )
    voltwatt_site_agreement = (vw_sites.blind_conformant == vw_sites.true_conformant).mean()

    agreement_rows.append({
        "rating_basis": rating_basis,
        "n_intervals_voltvar": len(vvar_intervals),
        "voltvar_interval_category_agreement": voltvar_interval_agreement,
        "n_sites_voltvar": len(vvar_sites),
        "voltvar_site_10pct_rule_agreement": voltvar_site_agreement,
        "n_intervals_voltwatt": len(vw_intervals),
        "voltwatt_interval_agreement": voltwatt_interval_agreement,
        "n_sites_voltwatt": len(vw_sites),
        "voltwatt_site_10pct_rule_agreement": voltwatt_site_agreement,
    })

blind_vs_true_agreement = pd.DataFrame(agreement_rows)
print("Blind (net-meter, inferred circuits) vs true (ami_raw PV-only) conformance agreement:")
display(blind_vs_true_agreement)

# Confusion matrix for PRIMARY_RATING_BASIS, Volt-VAr category-level.
primary_basis = PRIMARY_RATING_BASIS
primary_intervals = blind_results[primary_basis]["vvar"].merge(
    true_results[primary_basis]["vvar"][["site_id", "t_stamp", "voltvar_status"]].rename(
        columns={"voltvar_status": "true_voltvar_status"}),
    on=["site_id", "t_stamp"], how="inner",
)
print(f"Volt-VAr confusion matrix (rows=blind, cols=true), rating_basis={primary_basis}:")
display(pd.crosstab(primary_intervals.voltvar_status, primary_intervals.true_voltvar_status))

# Same comparison, but collapsed onto the REDUCED categories that actually
# drive the site-level 10% rule (Q_adverse/Q_inactive/Q_significant_shortfall
# vs everything else). Intervals not scored on either side (Q_not_assessable/
# Q_not_exposed) are dropped first -- they were never candidates for the
# reduced rule in the first place, on either the blind or the true side, so
# folding them into "not reduced" would understate the agreement rate.
scored_intervals = primary_intervals.loc[
    ~primary_intervals.voltvar_status.isin(Q_NOT_SCORED_STATUSES)
    & ~primary_intervals.true_voltvar_status.isin(Q_NOT_SCORED_STATUSES)
].copy()
scored_intervals["blind_reduced"] = np.where(
    scored_intervals.voltvar_status.isin(Q_REDUCED_CATEGORIES),
    "reduced_nonconformant", "conformant_or_surplus",
)
scored_intervals["true_reduced"] = np.where(
    scored_intervals.true_voltvar_status.isin(Q_REDUCED_CATEGORIES),
    "reduced_nonconformant", "conformant_or_surplus",
)
REDUCED_ORDER = ["reduced_nonconformant", "conformant_or_surplus"]

reduced_confusion_counts = pd.crosstab(
    scored_intervals.blind_reduced, scored_intervals.true_reduced,
).reindex(index=REDUCED_ORDER, columns=REDUCED_ORDER, fill_value=0)
reduced_confusion_pct_of_total = (
    100 * reduced_confusion_counts / reduced_confusion_counts.values.sum()
)
reduced_confusion_pct_of_row = (
    100 * reduced_confusion_counts.div(reduced_confusion_counts.sum(axis=1), axis=0)
)

print(f"\nVolt-VAr REDUCED-category confusion matrix (rows=blind, cols=true), "
      f"rating_basis={primary_basis} -- {len(scored_intervals):,} of "
      f"{len(primary_intervals):,} intervals scored on both sides:")
print("Counts:")
display(reduced_confusion_counts)
print("Percent of all scored intervals:")
display(reduced_confusion_pct_of_total.round(2))
print("Percent within each blind row (of the blind label, where does the true label land):")
display(reduced_confusion_pct_of_row.round(2))


### 7.3 Reconciling against notebook 02's Volt-Watt result (steps 1-4)

Section 6's ~99.8% Volt-Watt `fleet_conformant_pct` and `02_conformance_curtailment_analysis.ipynb`'s
~31% headline are not the same metric. Four concrete differences, addressed below:

1. **Wrong baseline for comparison.** The ~31% figure is `vwg_result` -- the
   "response-supported" Volt-Watt metric (`conformance_voltwattghi`), whose
   denominator only counts an interval when a violation was *directly
   observed* or the GHI clear-sky counterfactual *proves* enough sun was
   available to test the response. `ami_conformance.build_voltwatt_sql`
   implements only the *other* original table -- `build_conformance_voltwatt.py`'s
   `_insert_sql_basic` (no GHI counterfactual). The correct original number
   to compare against is `vw_result` (**84.56%**), not `vwg_result` (31.1%).
   There is no GHI/clear-sky counterfactual pipeline for the synthetic AMI
   dataset, so the response-supported metric is shown below for reference
   only -- it cannot be replicated here.
2. **Scope window.** `SCOPE_MONTHS` above is 3 spring months
   (`2024-09/10/11`); the original's `vw_result`/`vwg_result` pull
   `CONFIG.years = (2024, 2025)`, roughly two years including summer, when
   PV-export-driven overvoltage is worst. Re-run below with every month
   actually landed locally (`WIDE_SCOPE_MONTHS = None`), checked directly
   against `ami_meter_residential` rather than assumed.
3. **Source circuits for V/P.** The original's `data` CTE restricts to
   `ts.is_pv = True` (true PV-metered circuits). Section 6's blind pass sums
   `V`/`P_kw` over the *blind-inferred* PV-bearing load circuits instead --
   intentional for testing the blind method, but not the apples-to-apples
   comparison. Section 7.2's "true" pass (`true_results`, using `ami_raw`'s
   `V_pv`/`P_pv_kw`) is the actual ground-truth analog to `vw_result`'s
   `is_pv=True` methodology -- surfaced below in the same format for the
   first time.
4. **No voltage sanity filter.** The original drops `voltage <= 0` or
   `>= 300` readings before scoring (`stage2_common.py`'s `V_MIN`/`V_MAX`).
   Added to the wide-scope re-run below.

Every table below is in the exact `conformance_metrics.fleet_result` format
notebook 02 itself uses, so the comparison is a direct read.

In [ ]:
# Reference: the two original Volt-Watt results, exactly as computed in
# 02_conformance_curtailment_analysis.ipynb (Sections 2 & 3, live Athena
# pull against the real Solar Analytics fleet, rating_basis="ac_capacity_kw").
# Hardcoded here -- notebook 02's own SQL isn't runnable from this notebook
# (different data source) -- for direct, same-format comparison only.
notebook02_reference = pd.DataFrame([
    {
        "source": "02_conformance_curtailment_analysis.ipynb (original Athena pipeline)",
        "metric": "Volt-Watt maximum-output conformance",
        "rating_basis": "ac_capacity_kw",
        "question": "Did measured P exceed the voltage-dependent maximum?",
        "denominator": "voltage-exposed intervals (V > 253 V)",
        "n_sites_contained_in_table": 16148,
        "n_assessed_sites": 5021,
        "n_intervals": 6832290,
        "n_nonconforming_intervals": 872922,
        "interval_nonconf_pct": 12.776419,
        "fleet_conformant_pct": 84.564828,
        "fleet_any_nonconf_pct": 28.679546,
        "note": "THE CORRECT BASELINE for Section 6's blind/true Volt-Watt result below.",
    },
    {
        "source": "02_conformance_curtailment_analysis.ipynb (original Athena pipeline)",
        "metric": "Volt-Watt response-supported result",
        "rating_basis": "ac_capacity_kw",
        "question": "Was enough solar power available to test the response, or was a violation directly observed?",
        "denominator": "response-supported intervals (GHI counterfactual)",
        "n_sites_contained_in_table": 16148,
        "n_assessed_sites": 1971,
        "n_intervals": 951407,
        "n_nonconforming_intervals": 872922,
        "interval_nonconf_pct": 91.750639,
        "fleet_conformant_pct": 31.100964,
        "fleet_any_nonconf_pct": 73.059361,
        "note": "NOT REPLICABLE here -- no GHI/clear-sky counterfactual pipeline exists "
                "for the synthetic AMI dataset. Shown for reference only.",
    },
])
print("Reference -- original pipeline results (notebook 02), same fleet_result format:")
display(notebook02_reference)

In [ ]:
from bms_sa_review.data_query.lib.conformance_metrics import fleet_result

# BLIND pass, narrow scope (SCOPE_MONTHS = Sep/Oct/Nov 2024, Section 6's own
# inferred PV-bearing circuits) -- same rows Section 6 already computed,
# reformatted into notebook 02's own fleet_result shape for direct comparison.
blind_vw_rows = []
for rating_basis in RATING_BASES:
    result = fleet_result(
        blind_results[rating_basis]["vw_site"],
        "Volt-Watt maximum-output conformance -- BLIND (inferred PV-bearing circuits)",
        "Did measured P (inferred PV-bearing circuit(s)) exceed the voltage-dependent maximum?",
        "voltage-exposed intervals (V > VW.V1)",
        table_site_count=len(eligible),
    )
    result.insert(1, "rating_basis", rating_basis)
    result.insert(1, "scope", SCOPE_LABEL)
    result.insert(0, "source", "05_conformance_net_meter_blind.ipynb (this notebook)")
    blind_vw_rows.append(result)

blind_vw_summary = pd.concat(blind_vw_rows, ignore_index=True)
print(f"Blind pass, {SCOPE_LABEL} -- notebook 02's own fleet_result format:")
display(blind_vw_summary)

In [ ]:
# TRUE pass, narrow scope -- Section 7.2's ground-truth PV-circuit signal
# (ami_raw.V_pv / P_pv_kw, the same is_pv=True-circuit source the original
# pipeline's data CTE uses). This is the closer apples-to-apples comparison
# to vw_result's own methodology than the blind pass above.
true_vw_rows = []
for rating_basis in RATING_BASES:
    result = fleet_result(
        true_results[rating_basis]["vw_site"],
        "Volt-Watt maximum-output conformance -- TRUE (ami_raw is_pv circuits)",
        "Did measured P (true PV circuit) exceed the voltage-dependent maximum?",
        "voltage-exposed intervals (V > VW.V1)",
        table_site_count=len(eligible),
    )
    result.insert(1, "rating_basis", rating_basis)
    result.insert(1, "scope", SCOPE_LABEL)
    result.insert(0, "source", "05_conformance_net_meter_blind.ipynb (this notebook)")
    true_vw_rows.append(result)

true_vw_summary = pd.concat(true_vw_rows, ignore_index=True)
print(f"True (ground-truth PV circuit) pass, {SCOPE_LABEL} -- notebook 02's own fleet_result format:")
display(true_vw_summary)

In [ ]:
# Step 2 -- which months are actually landed locally? Checked directly
# against the data, not assumed. Answers "is it just a matter of setting
# SCOPE_MONTHS = None" concretely: None resolves to no WHERE restriction at
# all (scope_sql(None) == "1 = 1"), so it uses whatever is physically on
# disk -- which may or may not be the original's full (2024, 2025) window.
landed_months = con.sql("""
    SELECT DISTINCT year, month FROM ami_meter_residential ORDER BY year, month
""").df()
display(landed_months)
print(f"{len(landed_months)} (year, month) partitions landed locally in "
      f"ami_meter_residential -- SCOPE_MONTHS = None uses ALL of these, "
      f"not just {SCOPE_LABEL}. If 2025 doesn't appear above, None still "
      "won't reach full parity with the original's (2024, 2025) pull -- "
      "that would need the local store rebuilt to extract 2025 months too, "
      "a Phase 4/5 synthetic_ami_creation job, not a notebook setting.")

In [ ]:
# Step 2 + Step 4 -- rerun the BLIND Volt-Watt pass over every landed month
# (WIDE_SCOPE_MONTHS = None, independent of the notebook's own SCOPE_MONTHS
# -- nothing above or below this cell is affected), with the 0 < V < 300
# sanity filter stage2_common.py applies before scoring (Section 6 has
# neither). Entirely in DuckDB SQL -- per the MemoryError fixes in Section
# 7.1, nothing interval-level is pulled into pandas; a full year for ~6,600
# sites is ~4x the frame size that already OOM'd there. Only the final,
# small per-site (exposed_count, nonconforming_count) table is pulled back.
WIDE_SCOPE_MONTHS = None
WIDE_SCOPE_WHERE = scope_sql(WIDE_SCOPE_MONTHS, "m")

wide_vw_rows = []
wide_results = {}
for rating_basis in RATING_BASES:
    ratings = eligible[["site_id", rating_basis]].rename(columns={rating_basis: "rating_kw"})
    ratings = ratings.loc[ratings.rating_kw.notna() & (ratings.rating_kw > 0)]
    con.register("_ratings_wide", ratings)

    inner_sql = f"""
        SELECT s.site_id, s.t_stamp, s.V, s.P_kw, r.rating_kw
        FROM (
            SELECT m.site_id, m.t_stamp, avg(m.V) AS V, sum(m.P_kw) AS P_kw
            FROM ami_meter_residential m
            INNER JOIN pv_bearing_circuits c USING (site_id, circuit_id)
            WHERE {WIDE_SCOPE_WHERE}
            GROUP BY m.site_id, m.t_stamp
        ) s
        JOIN _ratings_wide r USING (site_id)
        WHERE s.V > 0 AND s.V < 300
    """
    vw_sql = Conformance.build_voltwatt_sql(f"({inner_sql}) AS scored", rating_column="rating_kw")
    site_sql = f"""
        WITH vw AS ({vw_sql})
        SELECT site_id,
               count(nonconformance_voltwatt) AS exposed_count,
               sum(CASE WHEN nonconformance_voltwatt > 0 THEN 1 ELSE 0 END) AS nonconforming_count
        FROM vw
        GROUP BY site_id
    """
    wide_site_counts = con.sql(site_sql).df()
    con.unregister("_ratings_wide")

    wide_site = aggregate_sites(
        wide_site_counts, "nonconforming_count", "exposed_count", threshold=SITE_NONCONF_THRESHOLD,
    )
    wide_results[rating_basis] = wide_site

    result = fleet_result(
        wide_site,
        "Volt-Watt maximum-output conformance -- BLIND, WIDE SCOPE + V sanity filter",
        "Did measured P (inferred PV-bearing circuit(s)) exceed the voltage-dependent maximum?",
        "voltage-exposed intervals (V > VW.V1), 0 < V < 300 sanity-filtered",
        table_site_count=len(eligible),
    )
    result.insert(1, "rating_basis", rating_basis)
    result.insert(1, "scope", "all landed months (WIDE_SCOPE_MONTHS = None)")
    result.insert(0, "source", "05_conformance_net_meter_blind.ipynb (this notebook)")
    wide_vw_rows.append(result)

wide_vw_summary = pd.concat(wide_vw_rows, ignore_index=True)
print("Blind pass, ALL landed months, 0 < V < 300 sanity filter (steps 2 & 4) -- "
      "notebook 02's own fleet_result format:")
display(wide_vw_summary)

print("\nAll four rows side by side (original basic baseline, blind narrow, "
      "true narrow, blind wide+filtered):")
display(pd.concat([
    notebook02_reference.loc[notebook02_reference.metric == "Volt-Watt maximum-output conformance"],
    blind_vw_summary, true_vw_summary, wide_vw_summary,
], ignore_index=True)[[
    "source", "metric", "scope", "rating_basis", "denominator",
    "n_sites_contained_in_table", "n_assessed_sites", "n_intervals",
    "n_nonconforming_intervals", "interval_nonconf_pct",
    "fleet_conformant_pct", "fleet_any_nonconf_pct",
]])

## 8. Historical comparison -- Solar Analytics / Athena

The three-way comparison: (1) **blind** (Section 4, net-meter, inferred
circuits), (2) **true-synthetic** (Section 5.2, `ami_raw.P_pv_kw`/`Q_pv_kvar`),
and (3) the **historical, already-computed** result from your earlier Solar
Analytics conformance run, read straight out of Athena --
`Config.TABLES["conformance_voltvar"]`/`["conformance_voltwatt"]`
(`conformance_voltvar_v2_flex_included`/`conformance_voltwatt_v2_flex_included`
in `solar_analytics_iceberg`). Nothing is recomputed here; this only queries
what already exists, filtered to this notebook's `eligible` cohort by
`site_id` -- confirmed to be the real Solar Analytics id, not a synthetic
renumbering (`ami_metadata.build_site_level_query` pulls straight from
`meta_up23c` by `site_id`).

**Requires AWS SSO login in your own terminal first**
(`aws sso login --profile ciccada`) -- this cell cannot do that for you.



### 8.1 Check whether the table holds one methodology or several stacked
together

`fetch_stored_provenance`-style: group by every methodological column and
count. **One row back means the table is a single run** -- safe to aggregate
directly. **More than one row means several runs are stacked in the same
table** -- an unfiltered `SUM(...)` would silently combine them, which is
exactly what `data_query.lib.conformance_queries.fetch_vvar_site_year`/
`fetch_vw_site_year` do (no methodology filter at all) -- fine only when the
table truly holds one combination. The pull in 6.2 filters explicitly
regardless, so it's correct either way, but this cell is how you'd know
without that filter.


In [ ]:
from bms_sa_review.shared.ciccada_config import SAI, TABLES
from bms_sa_review.synthetic_ami_creation.lib import ami_athena as Athena

Athena.require_credentials()
print(Athena.credential_status())

VOLTVAR_TABLE = TABLES["conformance_voltvar"]
VOLTWATT_TABLE = TABLES["conformance_voltwatt"]

voltvar_provenance = Athena.aq(f"""
    SELECT rating_basis, empirical_limit_basis, capability_profile,
           voltage_aggregation, flex_selection,
           count(*) AS n_rows, count(DISTINCT site_id) AS n_sites,
           min(year) AS first_year, max(year) AS last_year
    FROM {VOLTVAR_TABLE}
    GROUP BY rating_basis, empirical_limit_basis, capability_profile, voltage_aggregation, flex_selection
""", database=SAI)
print(f"{VOLTVAR_TABLE} -- distinct methodology combinations "
      "(>1 row = several runs stacked together):")
display(voltvar_provenance)

voltwatt_provenance = Athena.aq(f"""
    SELECT rating_basis, voltage_aggregation, flex_selection,
           count(*) AS n_rows, count(DISTINCT site_id) AS n_sites,
           min(year) AS first_year, max(year) AS last_year
    FROM {VOLTWATT_TABLE}
    GROUP BY rating_basis, voltage_aggregation, flex_selection
""", database=SAI)
print(f"{VOLTWATT_TABLE} -- distinct methodology combinations:")
display(voltwatt_provenance)


### 8.2 The pull -- one explicit methodology, matching the synthetic side

Filtered to `PRIMARY_RATING_BASIS` and `CAPABILITY_PROFILE` (both set in the
bootstrap cell, both printed below so the exact combination pulled is never
implicit in a column name) plus the project defaults for the two settings
this notebook doesn't otherwise vary (`voltage_aggregation="avg"`,
`flex_selection="exclude"`, matching `AnalysisConfig`'s own defaults).
`HISTORICAL_MONTHS` mirrors `SCOPE_MONTHS` -- leave it as `SCOPE_MONTHS` to
test one month first, or set it to `None` (independently of the synthetic
side, which can stay scoped) to pull `HISTORICAL_YEARS` in full once you're
ready to compare against everything.


In [ ]:
HISTORICAL_YEARS = (2024, 2025)         # used only when HISTORICAL_MONTHS is None
HISTORICAL_MONTHS = None                 # None pulls all of HISTORICAL_YEARS -- set back to
                                          # SCOPE_MONTHS only once you've confirmed real
                                          # Solar Analytics data actually exists for that
                                          # specific calendar month for this cohort
HISTORICAL_VOLTAGE_AGGREGATION = "avg"   # AnalysisConfig default -- not varied elsewhere here
HISTORICAL_FLEX_SELECTION = "include"    # AnalysisConfig default -- not varied elsewhere here


def historical_scope_sql(years, months):
    if months is None:
        year_list = ", ".join(str(y) for y in years)
        return f"year IN ({year_list})"
    pairs = [tuple(int(p) for p in m.split("-")) for m in months]
    clauses = " OR ".join(f"(year = {y} AND month = {m})" for y, m in pairs)
    return f"({clauses})"


print("Historical (Solar Analytics / Athena) pull -- methodology used:")
display(pd.DataFrame([
    {"setting": "years", "value": ", ".join(map(str, HISTORICAL_YEARS))},
    {"setting": "months (only used when HISTORICAL_MONTHS is not None)",
     "value": "all of the years above" if HISTORICAL_MONTHS is None else ", ".join(HISTORICAL_MONTHS)},
    {"setting": "rating_basis", "value": PRIMARY_RATING_BASIS},
    {"setting": "capability_profile (Volt-VAr only)", "value": CAPABILITY_PROFILE},
    {"setting": "voltage_aggregation", "value": HISTORICAL_VOLTAGE_AGGREGATION},
    {"setting": "flex_selection", "value": HISTORICAL_FLEX_SELECTION},
]))

# --- Diagnostic: how many of THIS notebook's eligible sites appear in each
# historical table AT ALL -- no year/month/methodology filter yet -- so a
# zero-row pull below is actually diagnosable instead of a second mystery.
for _tbl in (TABLES["conformance_voltvar"], TABLES["conformance_voltwatt"]):
    _diag = Athena.aq(f"""
        SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites,
               min(year) AS first_year, max(year) AS last_year
        FROM {_tbl}
        WHERE site_id IN ({eligible_site_ids})
    """, database=SAI, label=f"diag_{_tbl}")
    n_sites = int(_diag["n_sites"].iloc[0])
    if n_sites == 0:
        print(f"{_tbl}: NONE of this notebook's eligible sites appear in this table at all, "
              "in any year/month/methodology. The site cohort itself doesn't overlap this "
              "historical table -- check whether `eligible_site_ids` really are Solar Analytics "
              "site_ids this table has ever scored, independent of scope or methodology.")
    else:
        print(f"{_tbl}: {n_sites:,} of this notebook's eligible sites appear at all, "
              f"spanning {_diag['first_year'].iloc[0]}-{_diag['last_year'].iloc[0]} "
              f"({int(_diag['n_rows'].iloc[0]):,} rows). If the pull below is still 0, "
              "the year/month window or the methodology strings (rating_basis/"
              "capability_profile/voltage_aggregation/flex_selection) are the culprit -- "
              "compare against Section 8.1's own provenance table for the exact values "
              "actually present in this table.")

historical_vvar_raw = Athena.aq(f"""
    SELECT site_id,
           sum(Q_adverse_count) AS adverse,
           sum(Q_inactive_count) AS inactive,
           sum(Q_significant_shortfall_count) AS shortfall,
           sum(total_count) AS capability_assessable_count
    FROM {TABLES["conformance_voltvar"]}
    WHERE {historical_scope_sql(HISTORICAL_YEARS, HISTORICAL_MONTHS)}
      AND rating_basis = \'{PRIMARY_RATING_BASIS}\'
      AND capability_profile = \'{CAPABILITY_PROFILE}\'
      AND voltage_aggregation = \'{HISTORICAL_VOLTAGE_AGGREGATION}\'
      AND flex_selection = \'{HISTORICAL_FLEX_SELECTION}\'
      AND site_id IN ({eligible_site_ids})
    GROUP BY site_id
""", database=SAI, label="historical_voltvar_site")
historical_vvar_raw["reduced_nonconf_count"] = (
    historical_vvar_raw[["adverse", "inactive", "shortfall"]].sum(axis=1)
)
historical_vvar_site = aggregate_sites(
    historical_vvar_raw, "reduced_nonconf_count", "capability_assessable_count",
    threshold=SITE_NONCONF_THRESHOLD,
)

historical_vw_raw = Athena.aq(f"""
    SELECT site_id,
           sum(nonconformance_voltwatt_count) AS nonconforming_count,
           sum(total_count) AS exposed_count
    FROM {TABLES["conformance_voltwatt"]}
    WHERE {historical_scope_sql(HISTORICAL_YEARS, HISTORICAL_MONTHS)}
      AND rating_basis = \'{PRIMARY_RATING_BASIS}\'
      AND voltage_aggregation = \'{HISTORICAL_VOLTAGE_AGGREGATION}\'
      AND flex_selection = \'{HISTORICAL_FLEX_SELECTION}\'
      AND site_id IN ({eligible_site_ids})
    GROUP BY site_id
""", database=SAI, label="historical_voltwatt_site")
historical_vw_site = aggregate_sites(
    historical_vw_raw, "nonconforming_count", "exposed_count", threshold=SITE_NONCONF_THRESHOLD,
)

print(f"{len(historical_vvar_site):,} sites with historical Volt-VAr results, "
      f"{len(historical_vw_site):,} with historical Volt-Watt results, in this cohort.")
Athena.scan_report()

In [ ]:
print("Volt-VAr -- the one methodology combination actually stored:")
display(voltvar_provenance)
print("Volt-Watt -- the one methodology combination actually stored:")
display(voltwatt_provenance)
print(f"\nCell 8.2 is filtering for: rating_basis={PRIMARY_RATING_BASIS!r}, "
      f"capability_profile={CAPABILITY_PROFILE!r}, "
      f"voltage_aggregation={HISTORICAL_VOLTAGE_AGGREGATION!r}, "
      f"flex_selection={HISTORICAL_FLEX_SELECTION!r}")

In [ ]:
for _tbl in ("conformance_voltvar", "conformance_voltwattghi"):
    print(f"--- {_tbl} ---")
    display(Athena.describe(_tbl))
    _diag = Athena.aq(f"""
        SELECT count(*) AS n_rows, count(DISTINCT site_id) AS n_sites,
               min(year) AS first_year, max(year) AS last_year
        FROM {_tbl}
        WHERE site_id IN ({eligible_site_ids})
    """, database=SAI, label=f"diag_legacy_{_tbl}")
    display(_diag)

### 8.3 Three-way site-level agreement

Same site-level 10% rule, same `conformant` flag, from all three sources --
joined on `site_id`, compared pairwise and all together, for
`PRIMARY_RATING_BASIS` (the single basis this section pulled).


In [ ]:
def three_way_agreement(blind_site, true_site, historical_site, label):
    """
    site_id is coerced to a plain int64 on all three sides before merging --
    the Athena pull comes back through awswrangler/pandas, which can hand
    back a nullable Int64 (or, for an all-NULL/empty result, object) dtype
    for a BIGINT column, while the local blind/true frames carry whatever
    DuckDB's `.df()` produced (plain int64). Two differently-typed integer
    columns can silently match zero rows in a merge instead of raising, which
    is exactly how a real mismatch here turns into empty output with no
    error at all -- so the dtypes are normalised first and also printed,
    to make a *future* zero-row merge diagnosable without guessing again.
    """
    blind = blind_site[["site_id", "conformant"]].rename(
        columns={"conformant": "blind_conformant"})
    blind["site_id"] = blind["site_id"].astype("int64")
    true = true_site[["site_id", "conformant"]].rename(
        columns={"conformant": "true_conformant"})
    true["site_id"] = true["site_id"].astype("int64")
    historical = historical_site[["site_id", "conformant"]].rename(
        columns={"conformant": "historical_conformant"})
    if not historical.empty:
        historical["site_id"] = historical["site_id"].astype("int64")

    print(f"  [{label}] site counts -- blind: {len(blind):,}, true: {len(true):,}, "
          f"historical: {len(historical):,}")

    if historical.empty:
        print(
            f"  [{label}] historical_site has 0 rows -- the Athena pull in Section 8.2 "
            "returned nothing for this cohort/scope, so the three-way merge below is "
            "empty by construction (not a merge bug). The likeliest cause: "
            "HISTORICAL_MONTHS is mirroring SCOPE_MONTHS (set in the bootstrap cell for "
            "the synthetic side), and the real Solar Analytics data for these site_ids "
            "may simply not exist for that one calendar month. Try setting "
            "HISTORICAL_MONTHS = None in Section 8.2 (pulls all of HISTORICAL_YEARS "
            "instead) and re-running from there. If it is still 0 after that, re-check "
            "Section 8.1's methodology-combination table -- PRIMARY_RATING_BASIS/"
            "CAPABILITY_PROFILE/HISTORICAL_VOLTAGE_AGGREGATION/HISTORICAL_FLEX_SELECTION "
            "must exactly match one of the combinations actually present in "
            f"{TABLES['conformance_voltvar' if label == 'Volt-VAr' else 'conformance_voltwatt']}."
        )

    merged = blind.merge(true, on="site_id", how="inner").merge(historical, on="site_id", how="inner")
    print(f"  [{label}] merged (all three sources): {len(merged):,} sites")

    return pd.DataFrame([{
        "mechanism": label,
        "n_sites_all_three": len(merged),
        "blind_vs_true_agreement": (merged.blind_conformant == merged.true_conformant).mean(),
        "blind_vs_historical_agreement": (merged.blind_conformant == merged.historical_conformant).mean(),
        "true_vs_historical_agreement": (merged.true_conformant == merged.historical_conformant).mean(),
        "all_three_agree": (
            (merged.blind_conformant == merged.true_conformant)
            & (merged.true_conformant == merged.historical_conformant)
        ).mean(),
    }])


three_way = pd.concat([
    three_way_agreement(
        blind_results[primary_basis]["vvar_site"], true_results[primary_basis]["vvar_site"],
        historical_vvar_site, "Volt-VAr",
    ),
    three_way_agreement(
        blind_results[primary_basis]["vw_site"], true_results[primary_basis]["vw_site"],
        historical_vw_site, "Volt-Watt",
    ),
], ignore_index=True)
print(f"Three-way site-level 10%-rule agreement, rating_basis={primary_basis}, "
      f"capability_profile={CAPABILITY_PROFILE}:")
three_way